# Auditoria de datasets PHMD

Este notebook recorre los datasets descargados a Drive y produce, para cada uno, un conjunto de metricas suficientes para decidir su rol experimental en el TFM. La auditoria es el paso previo a la harmonizacion y al preentrenamiento: determina que datasets entran al SSL, cuales se reservan para evaluacion downstream y cuales se descartan.

Las cuatro categorias en uso (ver `CLAUDE.md` sec. 4) son:

- **PRETRAIN_SOURCE**: dataset que participa en el SSL multi-dataset. No se evalua downstream bajo ninguna circunstancia.
- **TRANSFER_TARGET**: dataset reservado por completo fuera del pretraining. Solo se usa en evaluacion downstream para medir transferencia real.
- **DROP**: dataset incompatible con el pipeline (longitud insuficiente, padding_ratio inaceptable, calidad inviable).
- **EXCLUDED**: dataset que no se puede procesar por un bug externo documentado.

Por cada dataset el notebook calcula:

1. **Forma y volumen**: numero de unidades por split, longitudes, numero de canales, RAM estimada.
2. **Calidad**: NaN, canales constantes, rango global, outliers, drift entre unidades.
3. **Etiquetas**: lista explicita de `target_candidates`, target seleccionado y policy aplicada, tipo de tarea, cobertura, balance o rango.
4. **Granularidad temporal**: `time_col`, `sampling_rate`, delta temporal, irregularidad, gaps, timestamps duplicados, monotonicidad por unidad.
5. **Post-ventaneo estimado** para `W=512, S=256, P=16`: numero de ventanas, padding_ratio, ventanas por unidad, tokens validos, tamano estimado en MB, tiempo cubierto por la ventana.
6. **Decision de rol**: PRETRAIN_SOURCE / TRANSFER_TARGET / DROP / EXCLUDED con razon explicita.

Outputs:

- `MyDrive/fm_fl_phmd/audit/<NOMBRE>.json` (uno por dataset, reentrante).
- `MyDrive/fm_fl_phmd/audit/plots/<NOMBRE>.png` (panel visual).
- `results/audit/audit_report.csv` (tabla agregada).
- `results/audit/audit_report.md` (legible para apendice del TFM).
- `results/audit/audit_groups.json` (dataset -> cliente FL con role).
- `results/audit/audit_summary.json` (conteos cerrados, sanity checks).

## Setup

Montamos Drive y lanzamos `colab_init.sh`. Si la salida no termina en `Setup OK`, ejecutar primero `setup/colab_bootstrap.ipynb`.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh

## 1. Configuracion

Rutas, hiperparametros del pipeline (W=512, S=256, P=16 fijos para el MVP) y umbrales heuristicos. Los umbrales DROP siguen `CLAUDE.md` sec. 12: si tras estimar el ventaneo `padding_ratio > 0.8` o `n_windows < 100` el dataset va a DROP.

In [ ]:
from pathlib import Path

# Version de la auditoria. Sube si cambias la semantica de las metricas o de
# los overrides. Los JSONs guardados se invalidan cuando audit_version != AUDIT_VERSION.
AUDIT_VERSION = '2.3'

# Rutas en Drive
DRIVE_ROOT = Path('/content/drive/MyDrive/fm_fl_phmd')
RAW_DIR    = DRIVE_ROOT / 'raw'
MANIFEST   = RAW_DIR / '_manifest.json'
AUDIT_DIR  = DRIVE_ROOT / 'audit'
PLOTS_DIR  = AUDIT_DIR / 'plots'
LOG_FILE   = DRIVE_ROOT / 'logs' / 'audit.log'

# Rutas en el repositorio (esto si va a git)
REPO         = Path('/content/fm_fl_phmd')
REPORT_DIR   = REPO / 'results' / 'audit'
REPORT_CSV   = REPORT_DIR / 'audit_report.csv'
REPORT_MD    = REPORT_DIR / 'audit_report.md'
GROUPS_JSON  = REPORT_DIR / 'audit_groups.json'
SUMMARY_JSON = REPORT_DIR / 'audit_summary.json'

# Cache local rapida
AUDIT_CACHE = Path('/content/_audit_cache')

for d in [AUDIT_DIR, PLOTS_DIR, LOG_FILE.parent, REPORT_DIR, AUDIT_CACHE / 'datasets']:
    d.mkdir(parents=True, exist_ok=True)

# Hiperparametros del pipeline (CLAUDE.md sec. 12, fijos en el MVP)
WINDOW_SIZE = 512
STRIDE      = 256
PATCH_SIZE  = 16
N_PATCHES   = WINDOW_SIZE // PATCH_SIZE   # 32

# Politica de cola del ventaneo.
#
# Desde v2.3 el default es 'pad': si una trayectoria con T > W deja un resto
# tras la ultima ventana completa, anadimos una ventana extra parcial con
# padding por la derecha. La justificacion de la decision esta en
# results/audit/tail_policy_comparison.json y en results/audit/tail_policy_diff_v22_v23.csv:
# PHM18 (TT primary) perdia el 23.2% de sus filas crudas con 'drop'.
# Las mascaras valid_time_mask y valid_patch_mask absorben el padding
# anadido (ya forman parte del contrato tensorial).
#
# 'drop' se mantiene como modo opcional historico (util para benchmarks
# legados o comparativa); no se recomienda como politica operacional.
TAIL_POLICY = 'pad'

# Umbrales
UMBRAL_DROP_NAN_MAX       = 50.0   # % NaN max por canal -> DROP
UMBRAL_DROP_PADDING_RATIO = 0.8    # padding > 0.8 -> DROP
UMBRAL_DROP_N_WINDOWS_MIN = 100    # < 100 ventanas tras ventaneo -> DROP
UMBRAL_DROP_LONGITUD_MIN  = 200    # mediana de longitud < 200 -> DROP
UMBRAL_DOMINANCIA_CLIENTE = 0.6    # warning si un cliente FL > 60% de tokens

MAX_UNIDADES_STATS = 200  # submuestreo para datasets gigantes

# -----------------------------------------------------------------------------
# Overrides manuales por dataset. Solo se anaden tras inspeccion explicita.
# -----------------------------------------------------------------------------
SUBSET_ID_OVERRIDE = {
    'CMAPSS': 'FD',     # FD001-FD004 son condiciones operativas distintas
}

TARGET_COL_OVERRIDE = {
    'UNIBO21': 'soc',   # state-of-charge, no es RUL
}

# Datasets que phmd no puede procesar por bugs externos documentados
EXCLUIDOS_PERMANENTES = {
    'CURVES': 'AssertionError interno en phmd al cargar el primer task',
    'PHM19':  'phmd usa DataFrame.append() (removido en pandas 2.0+)',
}

REUSE_PNG     = True
FORCE_REAUDIT = False  # poner True para reauditar todo aunque la version coincida

print(f'Audit version : {AUDIT_VERSION}')
print(f'Audit en      : {AUDIT_DIR}')
print(f'Reporte en    : {REPORT_DIR}')
print(f'Cache local   : {AUDIT_CACHE}')
print(f'W={WINDOW_SIZE}  S={STRIDE}  P={PATCH_SIZE}  N={N_PATCHES}  tail_policy={TAIL_POLICY}')
print(f'Overrides     : subset_id={list(SUBSET_ID_OVERRIDE)}, target={list(TARGET_COL_OVERRIDE)}')
print(f'REUSE_PNG={REUSE_PNG}  FORCE_REAUDIT={FORCE_REAUDIT}')


## 2. Roles experimentales propuestos

Lista inicial de **11 TRANSFER_TARGET** (uno por dominio principal, con target limpio y reconocimiento en literatura). Esta tabla es la entrada de la auditoria. El bucle calcula metricas reales y, en funcion de ellas, confirma o rebaja el role propuesto (un dataset PRETRAIN_SOURCE puede acabar como DROP si no pasa los umbrales).

In [ ]:
TRANSFER_TARGETS_PROPUESTOS = {
    'CMAPSS':     'benchmark estandar de RUL en literatura PHM',
    'CWRU':       'benchmark clasico de clasificacion de fallos en rodamientos',
    'PHM18':      'unico wind con target multiclase claro',
    'PHMAP23':    'clasificacion de 25 fallos en gearboxes, alta diversidad',
    'PHME20':     'RUL pequeno y limpio, reto PHM clasico',
    'PBCP16':     'clasificacion 5 clases balanceada, misc bien etiquetado',
    'CALCE_CS2':  'benchmark de RUL en baterias',
    'HSG18':      'unico hdd con target binario y balance razonable',
    'CBM14':      'unico naval con target accesible',
    'IEEE14':     'unico mosfets_power con target RUL',
    'CNCMILL18':  'unico cnc_milling con target wear claro',
}

DROPS_A_PRIORI = {
    'MOSFET11': 'mediana de longitud 1 muestra: dato tabular, no es serie temporal',
    'AHU21':    'mediana de longitud 109 muestras: insuficiente para W=512',
}

print(f'TRANSFER_TARGET propuestos: {len(TRANSFER_TARGETS_PROPUESTOS)}')
print(f'DROP a priori:              {len(DROPS_A_PRIORI)}')
print(f'EXCLUIDOS por bugs phmd:    {len(EXCLUIDOS_PERMANENTES)}')

## 3. Cliente FL por dominio

Mapping dataset -> dominio. Solo los PRETRAIN_SOURCE participan en clientes FL. Los TRANSFER_TARGET tienen dominio anotado a efectos de reporte pero no entran al corpus federado.

In [ ]:
DOMINIO_DE_DATASET = {
    'CWRU': 'bearings', 'MFPT': 'bearings', 'XJTU-SY': 'bearings',
    'JNUB': 'bearings', 'KAUG17': 'bearings', 'IMS': 'bearings',
    'UPM20': 'bearings', 'UPM23': 'bearings', 'PRONOSTIA': 'bearings',
    'LGB20': 'bearings', 'RRB23': 'bearings', 'SEUGB17': 'bearings',
    'UOC18': 'bearings',
    'CALCE_CS2': 'batteries', 'CALCE_CX2': 'batteries',
    'FCLB19':    'batteries', 'NB1':       'batteries',
    'NB14':      'batteries', 'UNIBO21':   'batteries',
    'CMAPSS': 'aero_engines', 'NCMAPSS': 'aero_engines',
    'CBM14': 'naval', 'CBMv3': 'naval',
    'PHMAP21': 'gearboxes', 'PHMAP23': 'gearboxes', 'ARAMIS20': 'gearboxes',
    'CNCMILL18': 'cnc_milling', 'NMILL': 'cnc_milling',
    'CESNASA15': 'capacitors',
    'MOSFET11': 'mosfets_power', 'IEEE14': 'mosfets_power',
    'PTRB19': 'transformers',
    'HSF15': 'hdd', 'HSG18': 'hdd',
    'AC16': 'compressor',
    'DFD15': 'drills',
    'AHU21': 'building_hvac',
    'PHM14': 'wind', 'PHM18': 'wind',
    'PHM10': 'phm_challenges', 'PHM15': 'phm_challenges',
    'PHM19': 'phm_challenges', 'PHME20': 'phm_challenges',
    'PHME24': 'phm_challenges', 'PPD18': 'phm_challenges',
    'CURVES': 'learning_curves',
    'OBDD17': 'misc', 'PBCP16': 'misc', 'PBHP16': 'misc',
    'DUS20':  'misc', 'GENDEM18': 'misc',
    'HIRFNASA15': 'misc', 'SSPSNASA15': 'misc',
}

def dominio_de(nombre):
    return DOMINIO_DE_DATASET.get(nombre, 'misc')

print(f'Datasets mapeados a dominio: {len(DOMINIO_DE_DATASET)}')

## 4. Manifest de descarga y plan de auditoria

Leemos `_manifest.json` (lo dejo `00_download_datasets.ipynb`), identificamos los descargados con `status='ok'`, separamos EXCLUDED y comparamos con los audits ya hechos. Reentrancia: si existe `audit/<nombre>.json` con `status='ok'`, se salta.

In [ ]:
import json

manifest = json.loads(MANIFEST.read_text())

descargados_ok = sorted([
    n for n, d in manifest['datasets'].items()
    if d.get('status') == 'ok'
])
print(f'Datasets descargados (status=ok): {len(descargados_ok)}')

# El criterio para marcar un dataset como "hecho" es:
#   - el JSON existe,
#   - su status es 'ok',
#   - y su audit_version coincide con AUDIT_VERSION.
# Si la version es menor (o no esta), se reaudita. FORCE_REAUDIT=True fuerza
# que todos pasen a pendientes aunque la version coincida.
pendientes = []
hechos     = []
reauditar_por_version = []
for nombre in descargados_ok:
    if nombre in EXCLUIDOS_PERMANENTES:
        continue
    audit_json = AUDIT_DIR / f'{nombre}.json'
    if not audit_json.exists():
        pendientes.append(nombre)
        continue
    try:
        entrada = json.loads(audit_json.read_text())
    except Exception:
        pendientes.append(nombre)
        continue
    if entrada.get('status') != 'ok':
        pendientes.append(nombre)
        continue
    if FORCE_REAUDIT:
        pendientes.append(nombre)
        reauditar_por_version.append((nombre, entrada.get('audit_version'), 'forzado'))
        continue
    if entrada.get('audit_version') != AUDIT_VERSION:
        pendientes.append(nombre)
        reauditar_por_version.append((nombre, entrada.get('audit_version'), AUDIT_VERSION))
        continue
    hechos.append(nombre)

print(f'EXCLUDED por bugs phmd : {sorted(EXCLUIDOS_PERMANENTES)}')
print(f'Ya auditados a v{AUDIT_VERSION} : {len(hechos)}')
print(f'Pendientes             : {len(pendientes)}')
if reauditar_por_version:
    print(f'  -> reauditados por bump de version ({len(reauditar_por_version)}):')
    for n, vieja, motivo in reauditar_por_version[:10]:
        print(f'     {n:18s} v={vieja!s:>5s} -> v{motivo}')
if pendientes:
    print(f'Primeros 10            : {pendientes[:10]}')


## 5. Carga del dataset

Copia el `.zip` desde Drive a `/root/.phmd/datasets/` (la ruta donde `phmd` realmente busca en runtime) y pide a `phmd` que use `/root/.phmd` como `cache_dir`. La descompresion ocurre en el SSD de la VM, no en Drive. Tras procesar cada dataset, `limpiar_cache_dataset` borra **solo** el `.zip` copiado y el subdirectorio extraido de ese dataset; el estado interno de `phmd` se preserva entre iteraciones (v2.2.1).

`phmd` guarda algunos zip con nombres distintos al ID (`JNUB -> JNUBearing.zip`, `MFPT -> MPFT.zip`, etc.). `OVERRIDE_FILE_PREFIX` mapea esos casos.

Workaround heredado: `inyectar_zipfile_en_phmd()` se llama antes de cada `Dataset(...)` para sortear el `NameError` de `zipfile` que algunos readers de `phmd` disparan (caso PRONOSTIA), con reintento si el primer `Dataset(...)` falla por esa causa.


In [ ]:
import os
import shutil
import sys
import zipfile
from pathlib import Path
from phmd import datasets as phmd_datasets
import phmd as _phmd_module

# ----------------------------------------------------------------------------
# Version del loader. Cell 32 hace assert sobre esto para abortar
# inmediatamente si Colab esta sirviendo una version vieja en memoria.
# ----------------------------------------------------------------------------
LOADER_VERSION = 'v2.2.2'

PHMD_HOME_CACHE = '/root/.phmd'

OVERRIDE_FILE_PREFIX = {
    'JNUB':       'JNUBearing',
    'MFPT':       'MPFT',
    'PHM19':      'PHM2019',
    'SSPSNASA15': 'SSPSNASA2015',
}


def inyectar_zipfile_en_phmd():
    """Inyecta zipfile en todos los submodulos phmd ya cargados.

    Workaround para PRONOSTIA y similares: algunos readers referencian
    zipfile sin importarlo. Reinjectamos antes de cada carga porque algunos
    submodulos se cargan diferidos al pedir el dataset por primera vez.
    """
    for nombre_mod, mod in list(sys.modules.items()):
        if nombre_mod.startswith('phmd') and mod is not None:
            try:
                setattr(mod, 'zipfile', zipfile)
            except Exception:
                pass


inyectar_zipfile_en_phmd()


def archivos_dataset(nombre):
    """Localiza los .zip del dataset en Drive."""
    drive_dir = RAW_DIR / 'datasets'
    if not drive_dir.exists():
        return []
    objetivo = OVERRIDE_FILE_PREFIX.get(nombre, nombre).lower()
    encontrados = []
    for p in drive_dir.iterdir():
        if not p.is_file():
            continue
        stem = p.stem.lower()
        name = p.name.lower()
        if (stem == objetivo
            or stem.startswith(objetivo + '_')
            or name.startswith(objetivo + '.')):
            encontrados.append(p); continue
        if stem.startswith(objetivo) and len(stem) > len(objetivo):
            siguiente = stem[len(objetivo)]
            if siguiente.isalpha():
                encontrados.append(p)
    return encontrados


def _listar_phmd_datasets_dir():
    """Devuelve la lista de ficheros y subdirs en /root/.phmd/datasets/.
    Solo para diagnostico cuando phmd falla."""
    d = Path(PHMD_HOME_CACHE) / 'datasets'
    if not d.exists():
        return ['<no existe /root/.phmd/datasets/>']
    entradas = []
    for p in sorted(d.iterdir()):
        if p.is_file():
            entradas.append(f'  archivo {p.name} ({p.stat().st_size} bytes)')
        elif p.is_dir():
            n_subitems = sum(1 for _ in p.iterdir())
            entradas.append(f'  dir     {p.name}/ ({n_subitems} items)')
    return entradas or ['  (vacio)']


def cargar_dataset(nombre, verbose=True):
    """Copia el zip del dataset a /root/.phmd/datasets/ y carga el primer
    task/fold con phmd. Devuelve (ds, splits, ficheros_copiados).

    Cambios v2.2.2 frente a v2.2.1:
      - Diagnostico verboso opcional: imprime ruta y tamano de cada copia.
      - Si phmd falla con "Dataset not found", lista lo que hay en
        /root/.phmd/datasets/ para depuracion.
      - El resto sigue igual que v2.2.1.
    """
    inyectar_zipfile_en_phmd()

    phmd_cache = Path(PHMD_HOME_CACHE) / 'datasets'
    phmd_cache.mkdir(parents=True, exist_ok=True)

    ficheros_drive = archivos_dataset(nombre)
    if not ficheros_drive:
        raise FileNotFoundError(f'No se encuentran ficheros del dataset {nombre} en Drive')

    ficheros_copiados = []
    for f in ficheros_drive:
        destino = phmd_cache / f.name
        try:
            tamano_origen = f.stat().st_size
        except Exception:
            tamano_origen = -1
        recopiar = (not destino.exists()) or (
            tamano_origen > 0 and destino.stat().st_size != tamano_origen
        )
        if recopiar:
            shutil.copy(f, destino)
        tamano_destino = destino.stat().st_size if destino.exists() else 0
        if tamano_destino == 0:
            raise IOError(f'Copia incompleta a {destino} (0 bytes). '
                          f'Drive probablemente se desmonto. Reintenta.')
        if verbose:
            estado = 'copiado' if recopiar else 'reutilizado'
            print(f'  [{estado}] {f.name}: {tamano_origen} -> {destino} ({tamano_destino} bytes)')
        ficheros_copiados.append(destino)

    # Carga con reintento al NameError de zipfile (PRONOSTIA).
    try:
        ds = phmd_datasets.Dataset(nombre, cache_dir=PHMD_HOME_CACHE)
    except NameError as ne:
        if 'zipfile' in str(ne).lower():
            inyectar_zipfile_en_phmd()
            ds = phmd_datasets.Dataset(nombre, cache_dir=PHMD_HOME_CACHE)
        else:
            raise
    except Exception as exc:
        # Diagnostico extra: si phmd no encuentra el dataset, listamos lo que
        # realmente hay en /root/.phmd/datasets/. Esto deja claro si phmd
        # esta esperando una ruta o nombre distinto.
        if 'not found' in str(exc).lower() or isinstance(exc, FileNotFoundError):
            print(f'\n  >>> DIAGNOSTICO phmd fallo para {nombre}:')
            print(f'  >>> error: {type(exc).__name__}: {exc}')
            print(f'  >>> contenido actual de {phmd_cache}:')
            for linea in _listar_phmd_datasets_dir():
                print(f'  >>> {linea}')
            print(f'  >>> phmd version: {getattr(_phmd_module, "__version__", "?")}')
            print(f'  >>> cache_dir pasado: {PHMD_HOME_CACHE}')
            print()
        raise

    if not getattr(ds, 'tasks', None):
        raise RuntimeError(f'phmd no expone tareas para {nombre}')
    fold = ds.tasks[0][0]
    if isinstance(fold, dict):
        splits = {k: v for k, v in fold.items() if v is not None}
    elif isinstance(fold, (tuple, list)):
        nombres_split = ['train', 'val', 'test'][:len(fold)]
        splits  = {n: v for n, v in zip(nombres_split, fold) if v is not None}
    else:
        splits = {'train': fold}
    return ds, splits, ficheros_copiados


def limpiar_cache_dataset(nombre, ficheros_copiados):
    """Limpia SOLO lo del dataset recien procesado: los .zip que copiamos
    y cualquier subdir que phmd haya extraido con el nombre del dataset o
    su override. Tambien limpia /content/_audit_cache (defensivo)."""
    for f in (ficheros_copiados or []):
        try:
            if f.exists() and f.is_file():
                f.unlink()
        except Exception:
            pass

    phmd_datasets_dir = Path(PHMD_HOME_CACHE) / 'datasets'
    nombres_posibles = {nombre, OVERRIDE_FILE_PREFIX.get(nombre, nombre)}
    for nombre_posible in nombres_posibles:
        candidato = phmd_datasets_dir / nombre_posible
        if candidato.exists() and candidato.is_dir():
            shutil.rmtree(candidato, ignore_errors=True)

    shutil.rmtree(AUDIT_CACHE, ignore_errors=True)
    (AUDIT_CACHE / 'datasets').mkdir(parents=True, exist_ok=True)


def limpiar_cache_local():
    """Compatibilidad retro: solo limpia /content/_audit_cache.
    NO toca /root/.phmd."""
    shutil.rmtree(AUDIT_CACHE, ignore_errors=True)
    (AUDIT_CACHE / 'datasets').mkdir(parents=True, exist_ok=True)


print(f'Helpers de carga {LOADER_VERSION} definidos:')
print(f'  phmd version       : {getattr(_phmd_module, "__version__", "?")}')
print(f'  cache real de phmd : {PHMD_HOME_CACHE}/datasets/  (copia + lectura)')
print(f'  cache temporal     : {AUDIT_CACHE}  (sin uso activo)')
print( '  workaround zipfile : pre-inyeccion + reintento al NameError (PRONOSTIA)')
print( '  cleanup            : limpiar_cache_dataset(nombre, ficheros)  por dataset')
print( '  diagnostico        : verbose=True imprime tamanos de copia; si phmd falla,')
print( '                       lista /root/.phmd/datasets/ y la version de phmd')


## 6. Deteccion de columnas y target candidates

Identifica `unit_col`, `time_col`, columna(s) target y senales. La novedad respecto a iteraciones previas es la lista explicita de `target_candidates` (no solo el primero detectado): cuando phmd expone varias tareas, las recogemos todas para que la decision sea trazable y no se elija silenciosamente el primer task (`CLAUDE.md` sec. 11).

In [ ]:
def detectar_columnas(ds, splits, nombre=None):
    """Detecta unit_col, time_col, target candidates, subset_id_col y signal_cols.

    Cambios v2.1:
      - Acepta `nombre` para aplicar SUBSET_ID_OVERRIDE y TARGET_COL_OVERRIDE.
      - Devuelve subset_id_col (puede ser None) y lo excluye de signal_cols.
      - Si TARGET_COL_OVERRIDE define un target para el dataset, lo deja
        primero en target_candidates y como target_col_seleccionado.
    """
    meta = getattr(ds, 'metadata', None) or getattr(ds, 'meta', None) or {}
    if not isinstance(meta, dict):
        meta = {}
    df = None
    for k in ('train', 'val', 'test'):
        if k in splits and splits[k] is not None and len(splits[k]) > 0:
            df = splits[k]; break
    if df is None:
        return {'unit_col': None, 'subset_id_col': None,
                'target_col_seleccionado': None,
                'target_candidates': [], 'time_col': None, 'signal_cols': []}
    cols = list(df.columns)

    unit_col = meta.get('unit_id') or meta.get('group_col') or meta.get('unit')
    if unit_col not in cols:
        unit_col = None
    if unit_col is None:
        for c in ('unit', 'unit_id', 'id', 'engine_id', 'machine_id',
                  'experiment', 'series_id', 'unit_number'):
            if c in cols:
                unit_col = c; break
        else:
            for c in cols:
                if df[c].dtype.kind in 'iuO' and df[c].nunique() < len(df) / 2:
                    unit_col = c; break

    time_col = meta.get('time_col') or meta.get('cycle') or meta.get('time')
    if time_col not in cols:
        time_col = None
    if time_col is None:
        for c in ('time', 'cycle', 'timestamp', 'time_in_cycles', 't', 'step'):
            if c in cols:
                time_col = c; break

    # subset_id_col: override manual primero; despues heuristica conservadora.
    subset_id_col = SUBSET_ID_OVERRIDE.get(nombre) if nombre else None
    if subset_id_col is not None and subset_id_col not in cols:
        subset_id_col = None

    candidates = []
    try:
        for t in (ds.tasks or []):
            nombre_tarea = getattr(t, 'name', None) or getattr(t, 'task_name', None)
            if nombre_tarea and str(nombre_tarea) in cols and str(nombre_tarea) not in candidates:
                candidates.append(str(nombre_tarea))
    except Exception:
        pass
    for c in ('rul', 'RUL', 'target', 'label', 'class', 'fault',
              'state', 'condition', 'anomaly', 'y', 'wear'):
        if c in cols and c not in candidates:
            candidates.append(c)

    # Override de target si el dataset esta en la tabla manual.
    target_override = TARGET_COL_OVERRIDE.get(nombre) if nombre else None
    if target_override and target_override in cols:
        if target_override in candidates:
            candidates.remove(target_override)
        candidates.insert(0, target_override)
        target_col_seleccionado = target_override
    else:
        target_col_seleccionado = candidates[0] if candidates else None

    # signal_cols: descartamos unit/time/target/candidates y, ahora tambien,
    # subset_id_col (es metadata, no senal).
    descartar = {unit_col, time_col, target_col_seleccionado, subset_id_col} | set(candidates)
    descartar = {c for c in descartar if c is not None}
    signal_cols = [c for c in cols
                   if c not in descartar and df[c].dtype.kind in 'fiub']
    return {
        'unit_col':                unit_col,
        'subset_id_col':           subset_id_col,
        'target_col_seleccionado': target_col_seleccionado,
        'target_candidates':       candidates,
        'time_col':                time_col,
        'signal_cols':             signal_cols,
    }


def _trajectory_id_series(df, split_name, columnas):
    """Devuelve una Series con el trajectory_id derivado para cada fila.

    Si hay subset_id_col, el id es '{split}_{subset_id}_{unit_id}'. Si no, es
    '{split}_{unit_id}'. Usar el split en el id evita confundir unidades con
    el mismo numero entre train/test/val (CMAPSS lo hace, por ejemplo).
    """
    unit_col      = columnas.get('unit_col')
    subset_id_col = columnas.get('subset_id_col')
    if unit_col is None:
        return None
    base = split_name + '__' + df[unit_col].astype(str)
    if subset_id_col and subset_id_col in df.columns:
        base = split_name + '__' + df[subset_id_col].astype(str) + '__' + df[unit_col].astype(str)
    return base.astype(str)


print('Detector de columnas v2.1 definido (con subset_id y target overrides).')


## 7. Metricas de forma y volumen

Numero de unidades por split, longitudes (`min/q25/mediana/q75/max`), numero de canales, RAM estimada al cargar el dataset entero como float32.

In [ ]:
import numpy as np


def metricas_forma_volumen(splits, columnas):
    """Forma y volumen agregados por dataset.

    Cambios v2.1:
      - Las "unidades" son trajectory_id (split + subset_id + unit_col), no
        unit_col crudo. En CMAPSS esto multiplica unidades por 4 FDs y dos
        splits originales (asi que ~8x la cuenta cruda original).
      - n_unidades_total queda como alias retro-compatible de
        n_trayectorias_total para no romper consumidores existentes.
    """
    unit_col      = columnas['unit_col']
    subset_id_col = columnas.get('subset_id_col')
    signal_cols   = columnas['signal_cols']
    out = {
        'n_canales':           len(signal_cols),
        'splits_predefinidos': set(splits.keys()) - {'train'} != set(),
        'splits_disponibles':  sorted(splits.keys()),
        'n_filas':             {},
        'n_unidades':          {},
        'n_trayectorias':      {},
        'subset_id_col':       subset_id_col,
    }
    longs = []
    ram_bytes = 0
    for sn, df in splits.items():
        if df is None: continue
        out['n_filas'][sn] = int(len(df))
        if unit_col and unit_col in df.columns:
            traj = _trajectory_id_series(df, sn, columnas)
            l = df.groupby(traj.values).size() if traj is not None else df.groupby(unit_col).size()
            n_uniq_unit = int(df[unit_col].nunique())
            out['n_unidades'][sn]     = n_uniq_unit
            out['n_trayectorias'][sn] = int(len(l))
            longs.extend(l.values.tolist())
        else:
            out['n_unidades'][sn]     = 1
            out['n_trayectorias'][sn] = 1
            longs.append(len(df))
        if signal_cols:
            ram_bytes += len(df) * len(signal_cols) * 4
    if longs:
        a = np.array(longs)
        out['longitudes'] = {
            'min':     int(a.min()),
            'q25':     int(np.percentile(a, 25)),
            'mediana': int(np.median(a)),
            'q75':     int(np.percentile(a, 75)),
            'max':     int(a.max()),
        }
    else:
        out['longitudes'] = {'min': 0, 'q25': 0, 'mediana': 0, 'q75': 0, 'max': 0}
    out['ram_estimada_gb']      = round(ram_bytes / 1e9, 3)
    out['n_trayectorias_total'] = int(sum(out['n_trayectorias'].values()))
    # alias retro-compatible. En v2.0 era el conteo de unit_col crudo, ahora es
    # el de trayectorias. El consumidor downstream (forma_volumen del piloto)
    # ya usa trajectory_id, asi que se alinean.
    out['n_unidades_total']     = out['n_trayectorias_total']
    out['n_filas_total']        = int(sum(out['n_filas'].values()))
    return out

print('Metricas A v2.1 definidas (grouping por trajectory_id).')


## 8. Metricas de calidad

Valores reales: NaN por canal (max y mediana), canales constantes, rango global, outliers (`|z|>5`), drift entre unidades (varianza y kurtosis de las medias por unidad). Submuestreo a `MAX_UNIDADES_STATS` cuando el dataset es muy grande para mantener el coste razonable.

In [ ]:
import numpy as np
import pandas as pd


def _muestrear_unidades(df, unit_col, max_unidades, seed=42):
    if unit_col is None or unit_col not in df.columns:
        return df
    unidades = df[unit_col].unique()
    if len(unidades) <= max_unidades:
        return df
    rng = np.random.default_rng(seed)
    sel = rng.choice(unidades, size=max_unidades, replace=False)
    return df[df[unit_col].isin(sel)]


def metricas_calidad(splits, columnas, max_unidades=MAX_UNIDADES_STATS):
    unit_col    = columnas['unit_col']
    signal_cols = columnas['signal_cols']
    if not signal_cols:
        return {'nan_pct_max': None, 'nan_pct_mediana': None,
                'canales_constantes': 0, 'canales_constantes_pct': 0.0,
                'outliers_pct': None, 'drift_varianza_medias': None,
                'drift_kurtosis_medias': None,
                'rango_min_max_global': None, 'submuestreado': False}
    dfs = [v for v in splits.values() if v is not None and len(v) > 0]
    if not dfs:
        return {}
    df_full = pd.concat(dfs, ignore_index=True)
    df_use  = _muestrear_unidades(df_full, unit_col, max_unidades)
    submuestreado = (len(df_use) < len(df_full))
    X = df_use[signal_cols]

    nan_pct = (X.isna().mean() * 100).round(2)
    varianzas = X.var(skipna=True)
    canales_constantes = int((varianzas.fillna(0) < 1e-10).sum())
    try:
        rmin = float(X.min(skipna=True).min())
        rmax = float(X.max(skipna=True).max())
    except Exception:
        rmin, rmax = None, None
    try:
        mu = X.mean(skipna=True)
        sigma = X.std(skipna=True).replace(0, np.nan)
        z = (X - mu) / sigma
        out_pct = float((z.abs() > 5).mean().mean() * 100)
    except Exception:
        out_pct = None
    drift_var = drift_kurt = None
    if unit_col and unit_col in df_use.columns:
        try:
            m = df_use.groupby(unit_col)[signal_cols].mean()
            drift_var  = float(m.var(skipna=True).mean())
            drift_kurt = float(m.kurtosis(skipna=True).mean())
        except Exception:
            pass
    return {
        'nan_pct_max':            float(nan_pct.max()) if len(nan_pct) else None,
        'nan_pct_mediana':        float(nan_pct.median()) if len(nan_pct) else None,
        'canales_constantes':     canales_constantes,
        'canales_constantes_pct': round(canales_constantes / len(signal_cols) * 100, 2),
        'outliers_pct':           round(out_pct, 4) if out_pct is not None else None,
        'drift_varianza_medias':  round(drift_var, 4)  if drift_var  is not None else None,
        'drift_kurtosis_medias':  round(drift_kurt, 4) if drift_kurt is not None else None,
        'rango_min_max_global':   [rmin, rmax],
        'submuestreado':          submuestreado,
    }

print('Metricas B definidas.')

## 9. Metricas de etiquetas y target policy

Determina tipo de target (clasificacion binaria/multiclase, RUL, regresion), numero de clases y balance si es clasificacion, rango si es RUL, cobertura. Documenta `target_candidates` y `target_policy` aplicada. Por defecto del MVP: `ultimo_valor_valido` para RUL, `etiqueta_unidad` para clasificacion estatica.

In [ ]:
import numpy as np
import pandas as pd


def _tipo_target(serie, nombre_col):
    s = serie.dropna()
    if len(s) == 0:
        return 'sin_etiqueta'
    nu = s.nunique()
    nl = (nombre_col or '').lower()
    if nl in ('rul', 'time_to_failure', 'ttf', 'remaining_useful_life'):
        return 'rul'
    if nl in ('anomaly', 'is_anomaly', 'outlier'):
        return 'anomaly'
    if s.dtype.kind == 'b':
        return 'classification_binary'
    if s.dtype.kind in 'iuO':
        if nu == 2: return 'classification_binary'
        if nu <= 100: return 'classification_multiclass'
        return 'regression'
    if s.dtype.kind == 'f':
        if nu == 2: return 'classification_binary'
        if nu <= 20 and np.allclose(s.values, s.round().values, equal_nan=True):
            return 'classification_multiclass'
        if s.min() >= 0 and s.max() > 10 and nu > 20:
            return 'regression_or_rul'
        return 'regression'
    return 'desconocido'


def metricas_etiquetas(ds, splits, columnas):
    target_col = columnas['target_col_seleccionado']
    candidates = columnas['target_candidates']
    tareas_phmd = []
    try:
        for t in (ds.tasks or []):
            tareas_phmd.append(str(getattr(t, 'name', None) or
                                   getattr(t, 'task_name', None) or
                                   t.__class__.__name__))
    except Exception:
        pass
    out = {
        'tareas_phmd':             tareas_phmd,
        'target_col_seleccionado': target_col,
        'target_candidates':       candidates,
        'tipo_target':             'sin_etiqueta',
        'target_policy':           None,
        'n_clases':                None,
        'balance_clases':          None,
        'rul_min':                 None,
        'rul_max':                 None,
        'label_coverage_pct':      0.0,
    }
    if not target_col:
        return out
    valores = []
    for df in splits.values():
        if df is None or target_col not in df.columns:
            continue
        valores.append(df[target_col])
    if not valores:
        return out
    serie = pd.concat(valores, ignore_index=True)
    out['label_coverage_pct'] = round(float(serie.notna().mean() * 100), 2)
    tipo = _tipo_target(serie, target_col)
    out['tipo_target'] = tipo
    if tipo in ('classification_binary', 'classification_multiclass'):
        counts = serie.dropna().value_counts()
        out['n_clases'] = int(len(counts))
        if len(counts) >= 2 and counts.max() > 0:
            out['balance_clases'] = round(float(counts.min() / counts.max()), 4)
        out['target_policy'] = 'etiqueta_unidad'
    elif tipo in ('rul', 'regression_or_rul', 'regression'):
        s = serie.dropna()
        if len(s):
            out['rul_min'] = round(float(s.min()), 4)
            out['rul_max'] = round(float(s.max()), 4)
        out['target_policy'] = 'ultimo_valor_valido'
    else:
        out['target_policy'] = 'no_aplicable'
    return out

print('Metricas C definidas.')

## 10. Metricas temporales

Si el dataset tiene `time_col`, calculamos por unidad la diferencia entre instantes consecutivos (`delta`) y derivamos:

- `sampling_rate_hz`: solo si el delta es regular (`1 / delta_mediano`).
- `delta_mediano`, `delta_std`, `delta_q25`, `delta_q75`.
- `irregularidad`: coeficiente de variacion del delta (`std/mediana`).
- `gaps_n`, `gaps_max`: saltos `> 3 * delta_mediano`.
- `timestamps_duplicados`: filas con `time_value` repetido dentro de una unidad.
- `monotonicidad_unidades_pct`: % de unidades donde el tiempo es estrictamente creciente.

Si no hay `time_col`, los valores quedan a `None` y el `window_time_seconds` no se podra calcular: la interpretacion temporal de W=512 queda indefinida (`CLAUDE.md` sec. 12).

In [ ]:
import numpy as np
import pandas as pd


def metricas_temporales(splits, columnas, ds=None):
    out = {
        'time_col':                    columnas['time_col'],
        'sampling_rate_hz':            None,
        'sampling_rate_origen':        None,
        'delta_mediano':               None,
        'delta_std':                   None,
        'delta_q25':                   None,
        'delta_q75':                   None,
        'irregularidad':               None,
        'gaps_n':                      None,
        'gaps_max':                    None,
        'timestamps_duplicados':       None,
        'monotonicidad_unidades_pct': None,
    }
    # Intento de sampling_rate desde metadata phmd
    if ds is not None:
        meta = getattr(ds, 'metadata', None) or getattr(ds, 'meta', None) or {}
        if isinstance(meta, dict):
            for k in ('sampling_rate', 'sampling_rate_hz', 'sample_rate', 'fs', 'frequency_hz'):
                if k in meta and meta[k]:
                    try:
                        out['sampling_rate_hz']     = float(meta[k])
                        out['sampling_rate_origen'] = 'metadata'
                        break
                    except (TypeError, ValueError):
                        pass
    time_col = columnas['time_col']
    unit_col = columnas['unit_col']
    if not time_col:
        return out
    dfs = [v for v in splits.values() if v is not None and len(v) > 0]
    if not dfs:
        return out
    df = pd.concat(dfs, ignore_index=True)
    if time_col not in df.columns:
        return out
    deltas = []
    monotonas = 0
    total_unidades = 0
    duplicados_total = 0
    grupos = df.groupby(unit_col) if (unit_col and unit_col in df.columns) else [(None, df)]
    for _, sub in grupos:
        if len(sub) < 2: continue
        total_unidades += 1
        t = sub[time_col].values
        if np.all(np.diff(t) > 0):
            monotonas += 1
        duplicados_total += int(pd.Series(t).duplicated().sum())
        d = np.diff(t)
        d = d[d > 0]
        deltas.extend(d.tolist())
    out['timestamps_duplicados'] = int(duplicados_total)
    if total_unidades > 0:
        out['monotonicidad_unidades_pct'] = round(100.0 * monotonas / total_unidades, 2)
    if deltas:
        arr = np.array(deltas, dtype=float)
        med = float(np.median(arr))
        std = float(np.std(arr))
        out['delta_mediano'] = round(med, 6)
        out['delta_std']     = round(std, 6)
        out['delta_q25']     = round(float(np.percentile(arr, 25)), 6)
        out['delta_q75']     = round(float(np.percentile(arr, 75)), 6)
        if med > 0:
            out['irregularidad'] = round(std / med, 4)
            mask_gap = arr > 3 * med
            out['gaps_n']   = int(mask_gap.sum())
            out['gaps_max'] = round(float(arr[mask_gap].max()) if mask_gap.any() else 0.0, 6)
        if out['sampling_rate_hz'] is None and med > 0 and (out['irregularidad'] or 0) < 0.05:
            out['sampling_rate_hz']     = round(1.0 / med, 6)
            out['sampling_rate_origen'] = 'inferido'
    return out

print('Metricas temporales definidas.')

## 11. Metricas post-ventaneo para W=512

Estima sin generar las ventanas reales, con `tail_policy='pad'` por defecto (v2.3):

- `n_windows` por split y total para `W=512, S=256`.
- `n_valid_patches_total`: contado **exactamente por ventana** (no derivado de `padding_ratio`). Una ventana completa contribuye `N_PATCHES`; una ventana parcial contribuye `ceil(valid_len/P)`.
- `padding_ratio`: muestras de padding totales / (n_windows * W). Se mantiene como diagnostico, pero no se usa para derivar patches.
- `windows_por_unidad`, `dropped_tail_rows` (debe ser 0 con `tail_policy=pad`).
- Metricas densas v2.3 para distinguir senal vs coste real:
  - `n_dense_temporal_patches = n_windows * N_PATCHES`
  - `n_dense_channel_patches  = n_dense_temporal_patches * n_canales`
  - `invalid_patch_ratio      = 1 - n_channel_patches / n_dense_channel_patches`
  - `dense_vs_valid_ratio     = n_dense_channel_patches / n_channel_patches`
- `estimated_size_mb`.
- `window_time_seconds`: `W / sampling_rate` si disponible, o `W * delta_mediano`.

Importante: la ventana fija en muestras no equivale a ventana fija en tiempo cuando los datasets tienen frecuencias de muestreo distintas. Esto se documenta como amenaza a la validez en la memoria del TFM.


In [ ]:
import numpy as np


def metricas_post_ventaneo(splits, columnas, temporal,
                            window_size=WINDOW_SIZE, stride=STRIDE, patch_size=PATCH_SIZE,
                            tail_policy=None):
    """Estima n_windows, padding y patches del ventaneo sin generar las ventanas.

    Politica operacional (desde v2.3): tail_policy='pad' por defecto. Una
    trayectoria con T > W y resto > 0 tras la ultima ventana completa anade una
    ventana parcial con padding por la derecha. La ventana extra contribuye
    ceil(valid_len/P) patches validos y window_size - valid_len muestras de
    padding. Los patches validos se cuentan exactamente por ventana, no
    derivando de padding_ratio.

    tail_policy='drop' descarta el resto. Util para reproducir el audit v2.2 o
    comparativas; no es la politica recomendada.

    Cambios estructurales heredados de v2.1+:
      - Agrupa por trajectory_id (split + subset_id + unit_col).
      - Excluye subset_id_col del conteo de canales.

    Cambios nuevos en v2.3:
      - Devuelve metricas densas: n_dense_temporal_patches, n_dense_channel_patches,
        invalid_patch_ratio, dense_vs_valid_ratio. Permiten distinguir senal util,
        padding y coste real del encoder channel-independent.
    """
    if tail_policy is None:
        tail_policy = TAIL_POLICY
    unit_col      = columnas['unit_col']
    subset_id_col = columnas.get('subset_id_col')
    signal_cols   = columnas['signal_cols']
    C = len(signal_cols)
    n_patches_por_ventana = window_size // patch_size
    if C == 0:
        return {'n_windows_total': 0, 'n_windows_por_split': {},
                'padding_ratio': None, 'windows_por_unidad': None,
                'n_patches_total': 0, 'n_valid_patches_total': 0,
                'n_tokens': 0, 'estimated_size_mb': 0.0,
                'window_time_seconds': None,
                'window_size': window_size, 'stride': stride, 'patch_size': patch_size,
                'tail_policy': tail_policy, 'dropped_tail_rows': 0,
                'n_dense_temporal_patches': 0, 'n_dense_channel_patches': 0,
                'invalid_patch_ratio': None, 'dense_vs_valid_ratio': None}
    n_windows_split       = {}
    n_windows_total       = 0
    padding_muestras      = 0
    longitudes_unidades   = []
    dropped_tail_rows     = 0
    n_valid_patches_total = 0
    for sn, df in splits.items():
        if df is None: continue
        if unit_col and unit_col in df.columns:
            traj = _trajectory_id_series(df, sn, columnas)
            if traj is not None:
                longs = df.groupby(traj.values).size().values
            else:
                longs = df.groupby(unit_col).size().values
        else:
            longs = np.array([len(df)])
        nw_split = 0
        for T in longs:
            T = int(T)
            longitudes_unidades.append(T)
            if T >= window_size:
                nw = (T - window_size) // stride + 1
                n_valid_patches_total += nw * n_patches_por_ventana
                ultima_w_end = (nw - 1) * stride + window_size
                resto = T - ultima_w_end
                if tail_policy == 'pad' and resto > 0:
                    nw += 1
                    n_valid_patches_total += int(np.ceil(resto / patch_size))
                    padding_muestras += (window_size - resto)
                elif tail_policy == 'drop':
                    dropped_tail_rows += resto
                nw_split += nw
            else:
                # Serie corta: una ventana parcial con padding al final.
                # Independiente de tail_policy.
                nw_split += 1
                n_valid_patches_total += int(np.ceil(T / patch_size))
                padding_muestras += (window_size - T)
        n_windows_split[sn] = nw_split
        n_windows_total   += nw_split
    padding_ratio = (padding_muestras / (n_windows_total * window_size)) if n_windows_total > 0 else None
    windows_por_unidad = n_windows_total / max(1, len(longitudes_unidades))
    n_patches_total = n_windows_total * n_patches_por_ventana

    # Metricas densas v2.3: lo que pesaria el corpus si no enmascaramos padding.
    n_dense_temporal_patches = n_windows_total * n_patches_por_ventana
    n_dense_channel_patches  = n_dense_temporal_patches * C
    n_channel_patches_valid  = n_valid_patches_total * C
    invalid_patch_ratio = (1.0 - (n_channel_patches_valid / n_dense_channel_patches)
                          if n_dense_channel_patches > 0 else None)
    dense_vs_valid_ratio = (n_dense_channel_patches / n_channel_patches_valid
                            if n_channel_patches_valid > 0 else None)

    estimated_size_mb = round(n_windows_total * n_patches_por_ventana * patch_size * C * 4 / 1e6, 2)
    window_time_seconds = None
    if temporal.get('sampling_rate_hz'):
        window_time_seconds = round(window_size / temporal['sampling_rate_hz'], 6)
    elif temporal.get('delta_mediano'):
        window_time_seconds = round(window_size * temporal['delta_mediano'], 6)
    return {
        'n_windows_total':         n_windows_total,
        'n_windows_por_split':     n_windows_split,
        'padding_ratio':           round(padding_ratio, 4) if padding_ratio is not None else None,
        'windows_por_unidad':      round(windows_por_unidad, 3),
        'n_patches_total':         n_patches_total,
        'n_valid_patches_total':   n_valid_patches_total,
        'n_tokens':                n_valid_patches_total,
        'n_dense_temporal_patches': n_dense_temporal_patches,
        'n_dense_channel_patches': n_dense_channel_patches,
        'invalid_patch_ratio':     round(invalid_patch_ratio, 4) if invalid_patch_ratio is not None else None,
        'dense_vs_valid_ratio':    round(dense_vs_valid_ratio, 4) if dense_vs_valid_ratio is not None else None,
        'estimated_size_mb':       estimated_size_mb,
        'window_time_seconds':     window_time_seconds,
        'window_size':             window_size,
        'stride':                  stride,
        'patch_size':              patch_size,
        'tail_policy':             tail_policy,
        'dropped_tail_rows':       dropped_tail_rows,
    }

print(f'Metricas post-ventaneo v2.3 definidas (tail_policy={TAIL_POLICY} por defecto, metricas densas separadas).')


## 12. Decision de role

Reglas, en orden de severidad (v2.2+):

1. `EXCLUDED` si esta en `EXCLUIDOS_PERMANENTES` (no se procesa).
2. `DROP` si esta en `DROPS_A_PRIORI`.
3. Reglas duras universales: `DROP` si `nan_max > 50%` o `padding_ratio > 0.8`. Ningun dataset, ni siquiera un TT propuesto, sobrevive a estas.
4. `TRANSFER_TARGET` si esta en `TRANSFER_TARGETS_PROPUESTOS`. La heuristica de longitud o numero de ventanas no descalifica TT (decision del usuario en `CLAUDE.md` sec. 4.bis: el rol experimental pesa mas que el tamano).
5. Para no-TT, las reglas de descarte por `mediana_longitud < 200` o `n_windows < 100` aplican.
6. `PRETRAIN_SOURCE` en cualquier otro caso.


In [ ]:
def decidir_role(nombre, forma, calidad, ventaneo):
    """Asigna role a partir de las metricas. Orden:

      1. EXCLUDED / DROPS_A_PRIORI (overrides manuales).
      2. Reglas duras universales: NaN extremo y padding extremo. Ningun
         dataset, ni siquiera un TT manual, sobrevive a estas.
      3. TRANSFER_TARGETS_PROPUESTOS: respetar la lista cerrada en
         CLAUDE.md sec.4.bis. La heuristica de longitud/n_windows no debe
         tumbar un benchmark elegido a mano (CMAPSS es el caso canonico:
         media de longitud baja por test trajectories cortas, pero el
         piloto demuestra que el pipeline lo procesa correctamente).
      4. Resto de reglas (longitud minima, n_windows minimo) solo aplican a
         datasets que no son TT.
    """
    if nombre in DROPS_A_PRIORI:
        return 'DROP', DROPS_A_PRIORI[nombre]
    nan_max   = (calidad or {}).get('nan_pct_max') or 0
    len_med   = (forma or {}).get('longitudes', {}).get('mediana') or 0
    padding   = (ventaneo or {}).get('padding_ratio')
    n_windows = (ventaneo or {}).get('n_windows_total') or 0
    # 2. Reglas duras
    if nan_max > UMBRAL_DROP_NAN_MAX:
        return 'DROP', f'{nan_max:.1f}% NaN maximo por canal (umbral {UMBRAL_DROP_NAN_MAX}%)'
    if padding is not None and padding > UMBRAL_DROP_PADDING_RATIO:
        return 'DROP', f'padding_ratio {padding:.2f} > {UMBRAL_DROP_PADDING_RATIO} con W={WINDOW_SIZE}'
    # 3. TT propuesto a mano: se respeta pese a longitudes cortas / pocas ventanas
    if nombre in TRANSFER_TARGETS_PROPUESTOS:
        return 'TRANSFER_TARGET', TRANSFER_TARGETS_PROPUESTOS[nombre]
    # 4. Reglas de DROP que solo aplican a no-TT
    if len_med < UMBRAL_DROP_LONGITUD_MIN:
        return 'DROP', f'mediana de longitud {len_med} < {UMBRAL_DROP_LONGITUD_MIN}'
    if n_windows < UMBRAL_DROP_N_WINDOWS_MIN:
        return 'DROP', f'solo {n_windows} ventanas estimadas (umbral {UMBRAL_DROP_N_WINDOWS_MIN})'
    return 'PRETRAIN_SOURCE', 'pasa filtros minimos de calidad y volumen'

print('Decision de role v2.3 definida (TT propuestos bypasean longitud/n_windows; reglas duras: NaN > 50%, padding > 0.8).')


## 13. Plot de inspeccion (4 paneles)

Un PNG por dataset: serie ejemplo, histograma de longitudes (con linea en W=512), heatmap de NaN por canal y distribucion del target. Util para inspeccion manual de casos limite.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np


def plot_dataset(nombre, splits, columnas, metricas, ruta_png):
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(f'{nombre}', fontsize=13, fontweight='bold')
    df = None
    for k in ('train', 'val', 'test'):
        if k in splits and splits[k] is not None and len(splits[k]) > 0:
            df = splits[k]; break
    unit_col    = columnas['unit_col']
    signal_cols = columnas['signal_cols']
    target_col  = columnas['target_col_seleccionado']
    tipo_target = metricas['etiquetas'].get('tipo_target')

    ax = axes[0, 0]
    if df is not None and unit_col and signal_cols and unit_col in df.columns:
        lens = df.groupby(unit_col).size()
        unidad = lens.sub(lens.median()).abs().idxmin()
        muestra = df[df[unit_col] == unidad]
        for col in signal_cols[:5]:
            ax.plot(range(len(muestra)), muestra[col].values, label=col, lw=0.7)
        ax.set_title(f'Serie ejemplo  unidad {unidad}  len={len(muestra)}')
        ax.legend(fontsize=6, ncol=2, loc='upper right')
    else:
        ax.text(0.5, 0.5, 'sin datos suficientes', ha='center', va='center', transform=ax.transAxes)

    ax = axes[0, 1]
    if df is not None and unit_col and unit_col in df.columns:
        todas = []
        for d in splits.values():
            if d is not None and unit_col in d.columns:
                todas.extend(d.groupby(unit_col).size().values.tolist())
        if todas:
            ax.hist(todas, bins=min(30, len(todas)), color='steelblue', edgecolor='k')
            ax.axvline(WINDOW_SIZE, color='red', linestyle='--', label=f'W={WINDOW_SIZE}')
            ax.set_title(f'Longitudes  {len(todas)} unidades')
            ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, 'sin unit_col', ha='center', va='center', transform=ax.transAxes)

    ax = axes[1, 0]
    if df is not None and signal_cols:
        nan_pct = (df[signal_cols].isna().mean() * 100).values.reshape(1, -1)
        im = ax.imshow(nan_pct, aspect='auto', cmap='Reds', vmin=0, vmax=100)
        ax.set_xticks(range(len(signal_cols)))
        ax.set_xticklabels(signal_cols, rotation=45, ha='right', fontsize=5)
        ax.set_yticks([])
        ax.set_title('% NaN por canal')
        plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    else:
        ax.text(0.5, 0.5, 'sin senales', ha='center', va='center', transform=ax.transAxes)

    ax = axes[1, 1]
    if df is not None and target_col and target_col in df.columns:
        s = df[target_col].dropna()
        if len(s) > 0:
            if tipo_target and 'classification' in tipo_target:
                cnt = s.value_counts().sort_index()
                ax.bar(range(len(cnt)), cnt.values, color='seagreen')
                ax.set_xticks(range(len(cnt)))
                ax.set_xticklabels([str(x) for x in cnt.index], rotation=45, fontsize=7)
                ax.set_title(f'Clases ({len(cnt)})')
            elif tipo_target == 'rul':
                ax.hist(s.values, bins=30, color='coral', edgecolor='k')
                ax.set_title(f'RUL  rango [{s.min():.1f}, {s.max():.1f}]')
            else:
                ax.hist(s.values, bins=30, color='gray', edgecolor='k')
                ax.set_title(f'Target ({tipo_target})')
    else:
        ax.text(0.5, 0.5, 'sin target', ha='center', va='center', transform=ax.transAxes)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(ruta_png, dpi=85, bbox_inches='tight')
    plt.close(fig)

print('Plot definido.')

## 14. Persistencia y logging

In [ ]:
import json
from datetime import datetime, timezone


def guardar_audit(nombre, datos):
    ruta = AUDIT_DIR / f'{nombre}.json'
    tmp  = ruta.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(datos, indent=2, sort_keys=True, default=str))
    tmp.replace(ruta)


def log(msg):
    linea = f'[{datetime.now(timezone.utc).isoformat(timespec="seconds")}] {msg}'
    print(linea)
    with LOG_FILE.open('a') as f:
        f.write(linea + '\n')

print('Helpers de persistencia definidos.')

## 15. Bucle principal

Por cada dataset pendiente: cargar, detectar columnas, calcular las 5 secciones de metricas, decidir role, plot, guardar JSON, limpiar cache. Si algo falla, anotamos y seguimos.

In [ ]:
import time
import traceback

# Guard: aborta inmediatamente si Colab esta sirviendo la cell 12 vieja en
# memoria (el git pull en colab_init.sh ya recogio la version nueva pero
# Colab no recarga el notebook ya abierto; hay que hacer File -> Revert).
try:
    _v = LOADER_VERSION
except NameError:
    raise RuntimeError(
        'cell 12 no esta a dia: LOADER_VERSION no esta definido. '
        'Has hecho git pull pero Colab esta usando una version cacheada del '
        'notebook. Cierra esta pestana, abre el notebook de nuevo y vuelve '
        'a ejecutar todas las celdas desde la 1.'
    )
if _v != 'v2.2.2':
    raise RuntimeError(
        f'cell 12 es {_v}, se esperaba v2.2.2. Pulla el repo y haz File -> '
        f'Revert sobre el notebook antes de seguir.'
    )

log(f'Inicio del bucle v{AUDIT_VERSION} (loader {LOADER_VERSION}). '
    f'Pendientes: {len(pendientes)}. Ya hechos: {len(hechos)}.')

for i, nombre in enumerate(pendientes, 1):
    log(f'[{i}/{len(pendientes)}] {nombre} ...')
    t0 = time.time()
    datos = {
        'nombre':        nombre,
        'dominio':       dominio_de(nombre),
        'ts':            datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'status':        'ok',
        'window_size':   WINDOW_SIZE,
        'stride':        STRIDE,
        'patch_size':    PATCH_SIZE,
        'audit_version': AUDIT_VERSION,
        'loader_version': LOADER_VERSION,
        'tail_policy':   TAIL_POLICY,
    }
    ficheros_copiados = None
    try:
        ds, splits, ficheros_copiados = cargar_dataset(nombre, verbose=True)
        columnas   = detectar_columnas(ds, splits, nombre=nombre)
        datos['columnas']             = columnas
        datos['subset_id_col_detected'] = columnas.get('subset_id_col')
        datos['forma']     = metricas_forma_volumen(splits, columnas)
        datos['calidad']   = metricas_calidad(splits, columnas)
        datos['etiquetas'] = metricas_etiquetas(ds, splits, columnas)
        datos['temporal']  = metricas_temporales(splits, columnas, ds=ds)
        datos['ventaneo']  = metricas_post_ventaneo(
            splits, columnas, datos['temporal'], tail_policy=TAIL_POLICY,
        )
        role, razon = decidir_role(nombre, datos['forma'], datos['calidad'], datos['ventaneo'])
        datos['role']       = role
        datos['role_razon'] = razon
        ruta_png = PLOTS_DIR / f'{nombre}.png'
        if REUSE_PNG and ruta_png.exists():
            datos['plot']        = str(ruta_png.relative_to(DRIVE_ROOT))
            datos['plot_reused'] = True
        else:
            try:
                plot_dataset(nombre, splits, columnas, datos, ruta_png)
                datos['plot']        = str(ruta_png.relative_to(DRIVE_ROOT))
                datos['plot_reused'] = False
            except Exception as e:
                datos['plot_error'] = f'{type(e).__name__}: {e}'
        datos['elapsed_s'] = round(time.time() - t0, 1)
        guardar_audit(nombre, datos)
        log(f'  -> {role} ({razon}) [{datos["elapsed_s"]} s, '
            f'subset_id_col={columnas.get("subset_id_col")}, '
            f'plot_reused={datos.get("plot_reused")}]')
    except Exception as e:
        datos['status']    = 'failed'
        datos['error']     = f'{type(e).__name__}: {e}'
        datos['traceback'] = traceback.format_exc()
        datos['elapsed_s'] = round(time.time() - t0, 1)
        guardar_audit(nombre, datos)
        log(f'  -> FALLO: {datos["error"]}')
    finally:
        try:
            limpiar_cache_dataset(nombre, ficheros_copiados)
        except Exception as e:
            log(f'  (warning: cleanup de cache fallo: {e})')

    # Log de espacio en disco cada 10 datasets
    if i % 10 == 0:
        try:
            import shutil as _sh
            _t, _u, _f = _sh.disk_usage('/content')
            log(f'  [disco /content tras {i}] used={_u/1e9:.1f} GB '
                f'free={_f/1e9:.1f} GB de {_t/1e9:.0f} GB')
        except Exception:
            pass

log(f'Bucle v{AUDIT_VERSION} (loader {LOADER_VERSION}) terminado.')


## 16. Agregacion al CSV

Recoge todos los JSONs (incluyendo EXCLUDED como filas explicitas) y construye `audit_report.csv` con las columnas mas utiles para inspeccion y decision. Esta version (v2.3) anade:

- `n_dense_temporal_patches`, `n_dense_channel_patches`, `invalid_patch_ratio`, `dense_vs_valid_ratio` (metricas densas para distinguir senal vs coste).
- `tail_policy` por dataset.
- Snapshot del CSV previo (v2.2) al inicio, para computar el diff agregado.
- Asserts de cierre: roles 36/11/4/2, CMAPSS=TT primary, lista TT exacta, padding sin extremos en PS/TT, `audit_version` y `tail_policy` homogeneos.
- Diff agregado `tail_policy_diff_v22_v23.csv/json` con `delta_windows`, `delta_channel_patches_pct` y `padding` antes/despues para todos los PS+TT.

Si algun assert falla, el resto del notebook no se ejecuta (no se sobreescribiran `audit_report.md`, `audit_summary.json` ni `audit_groups.json`).


In [ ]:
import json
import math
from datetime import datetime, timezone
import pandas as pd

# Politica de shard vigente (debe coincidir con la harmonizacion).
SHARD_MAXCOUNT = 5000

# Evaluation tier solo aplica a TRANSFER_TARGET. Se mantiene aparte de role:
# describe la prioridad experimental, no la categoria del dataset.
EVALUATION_TIER = {
    'CWRU':      'primary',
    'HSG18':     'primary',
    'CALCE_CS2': 'primary',
    'PHM18':     'primary',
    'PBCP16':    'primary',
    'IEEE14':    'primary',
    'CMAPSS':    'primary',     # con cautela: rango RUL con valores negativos
    'CBM14':     'secondary',
    'CNCMILL18': 'secondary',
    'PHME20':    'secondary',
    'PHMAP23':   'secondary',
}


def _safe(d, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict): return default
        cur = cur.get(k)
        if cur is None: return default
    return cur


def _est_shards(d, n_windows_total):
    """Estima shards sumando ceil(n_windows_split / SHARD_MAXCOUNT) por split."""
    n_split = _safe(d, 'ventaneo', 'n_windows_por_split') or {}
    if isinstance(n_split, dict) and n_split:
        return int(sum(math.ceil(v / SHARD_MAXCOUNT) for v in n_split.values() if v and v > 0))
    if n_windows_total and n_windows_total > 0:
        return int(math.ceil(n_windows_total / SHARD_MAXCOUNT))
    return None


# ---------------------------------------------------------------------------
# 1. Snapshot del audit_report.csv ANTES de regenerarlo. Necesario para el
#    diff v2.2 -> v2.3 al final.
# ---------------------------------------------------------------------------
df_audit_previous = None
prev_version_str  = None
if REPORT_CSV.exists():
    try:
        df_audit_previous = pd.read_csv(REPORT_CSV)
        if 'audit_version' in df_audit_previous.columns:
            prev_version_str = str(df_audit_previous['audit_version'].iloc[0])
        print(f'[diff] Snapshot audit previo: {len(df_audit_previous)} filas, version={prev_version_str}')
    except Exception as e:
        print(f'[diff] AVISO snapshot: {e}')
        df_audit_previous = None
else:
    print('[diff] No hay audit_report.csv previo en repo. No se generara diff.')


# ---------------------------------------------------------------------------
# 2. Construir df_audit (filas + pesos relativos).
# ---------------------------------------------------------------------------
filas = []

# EXCLUDED por bug phmd como entradas explicitas
for nombre, razon in EXCLUIDOS_PERMANENTES.items():
    filas.append({
        'dataset': nombre, 'dominio': dominio_de(nombre), 'role': 'EXCLUDED',
        'role_razon': razon, 'status': 'excluded',
        'audit_version': AUDIT_VERSION,
        'subset_id_col_detected': None,
        'tail_policy': None,
        'n_unidades_total': None, 'n_filas_total': None, 'n_canales': None,
        'len_mediana': None, 'median_length_lt_window_size': None,
        'tipo_target': None, 'target_col': None, 'target_policy': None,
        'target_candidates': None,
        'evaluation_tier': None,
        'n_clases': None, 'balance_clases': None,
        'rul_min': None, 'rul_max': None,
        'label_coverage_pct': None,
        'time_col': None, 'sampling_rate_hz': None, 'sampling_rate_origen': None,
        'delta_mediano': None, 'irregularidad': None, 'gaps_n': None,
        'monotonicidad_pct': None,
        'nan_pct_max': None, 'canales_const_pct': None,
        'n_windows': None, 'windows_por_unidad': None, 'padding_ratio': None,
        'n_valid_patches': None, 'n_tokens': None,
        'n_temporal_patches': None, 'n_channel_patches': None,
        'n_dense_temporal_patches': None, 'n_dense_channel_patches': None,
        'invalid_patch_ratio': None, 'dense_vs_valid_ratio': None,
        'estimated_size_mb': None, 'estimated_n_shards': None,
        'estimated_compute_weight': None,
        'dataset_weight_temporal_patches': None,
        'dataset_weight_channel_patches':  None,
        'window_time_seconds': None,
        'dropped_tail_rows': None,
        'plot': None, 'error': None,
    })

for fichero in sorted(AUDIT_DIR.glob('*.json')):
    d = json.loads(fichero.read_text())
    nombre        = d.get('nombre')
    n_valid       = _safe(d, 'ventaneo', 'n_valid_patches_total')
    n_canales     = _safe(d, 'forma', 'n_canales')
    n_windows     = _safe(d, 'ventaneo', 'n_windows_total')
    len_mediana   = _safe(d, 'forma', 'longitudes', 'mediana')

    n_temporal_patches = n_valid
    n_channel_patches  = (n_valid * n_canales) if (n_valid is not None and n_canales) else None
    median_lt_W        = (len_mediana is not None and len_mediana < WINDOW_SIZE)

    filas.append({
        'dataset':                          nombre,
        'dominio':                          d.get('dominio'),
        'role':                             d.get('role'),
        'role_razon':                       d.get('role_razon'),
        'status':                           d.get('status'),
        'audit_version':                    d.get('audit_version'),
        'subset_id_col_detected':           d.get('subset_id_col_detected'),
        'tail_policy':                      _safe(d, 'ventaneo', 'tail_policy') or d.get('tail_policy'),
        'n_unidades_total':                 _safe(d, 'forma', 'n_unidades_total'),
        'n_filas_total':                    _safe(d, 'forma', 'n_filas_total'),
        'n_canales':                        n_canales,
        'len_mediana':                      len_mediana,
        'median_length_lt_window_size':     median_lt_W,
        'tipo_target':                      _safe(d, 'etiquetas', 'tipo_target'),
        'target_col':                       _safe(d, 'etiquetas', 'target_col_seleccionado'),
        'target_policy':                    _safe(d, 'etiquetas', 'target_policy'),
        'target_candidates':                ';'.join(_safe(d, 'etiquetas', 'target_candidates') or []),
        'evaluation_tier':                  EVALUATION_TIER.get(nombre),
        'n_clases':                         _safe(d, 'etiquetas', 'n_clases'),
        'balance_clases':                   _safe(d, 'etiquetas', 'balance_clases'),
        'rul_min':                          _safe(d, 'etiquetas', 'rul_min'),
        'rul_max':                          _safe(d, 'etiquetas', 'rul_max'),
        'label_coverage_pct':               _safe(d, 'etiquetas', 'label_coverage_pct'),
        'time_col':                         _safe(d, 'temporal', 'time_col'),
        'sampling_rate_hz':                 _safe(d, 'temporal', 'sampling_rate_hz'),
        'sampling_rate_origen':             _safe(d, 'temporal', 'sampling_rate_origen'),
        'delta_mediano':                    _safe(d, 'temporal', 'delta_mediano'),
        'irregularidad':                    _safe(d, 'temporal', 'irregularidad'),
        'gaps_n':                           _safe(d, 'temporal', 'gaps_n'),
        'monotonicidad_pct':                _safe(d, 'temporal', 'monotonicidad_unidades_pct'),
        'nan_pct_max':                      _safe(d, 'calidad', 'nan_pct_max'),
        'canales_const_pct':                _safe(d, 'calidad', 'canales_constantes_pct'),
        'n_windows':                        n_windows,
        'windows_por_unidad':               _safe(d, 'ventaneo', 'windows_por_unidad'),
        'padding_ratio':                    _safe(d, 'ventaneo', 'padding_ratio'),
        'n_valid_patches':                  n_valid,
        'n_tokens':                         _safe(d, 'ventaneo', 'n_tokens'),
        'n_temporal_patches':               n_temporal_patches,
        'n_channel_patches':                n_channel_patches,
        'n_dense_temporal_patches':         _safe(d, 'ventaneo', 'n_dense_temporal_patches'),
        'n_dense_channel_patches':          _safe(d, 'ventaneo', 'n_dense_channel_patches'),
        'invalid_patch_ratio':              _safe(d, 'ventaneo', 'invalid_patch_ratio'),
        'dense_vs_valid_ratio':             _safe(d, 'ventaneo', 'dense_vs_valid_ratio'),
        'estimated_size_mb':                _safe(d, 'ventaneo', 'estimated_size_mb'),
        'estimated_n_shards':               _est_shards(d, n_windows),
        'estimated_compute_weight':         n_channel_patches,
        'dataset_weight_temporal_patches':  None,
        'dataset_weight_channel_patches':   None,
        'window_time_seconds':              _safe(d, 'ventaneo', 'window_time_seconds'),
        'dropped_tail_rows':                _safe(d, 'ventaneo', 'dropped_tail_rows'),
        'plot':                             d.get('plot'),
        'error':                            d.get('error'),
    })

df_audit = pd.DataFrame(filas)

# Pesos relativos sobre el corpus de pretraining (solo PRETRAIN_SOURCE).
ps_mask    = df_audit['role'] == 'PRETRAIN_SOURCE'
total_temp = float(df_audit.loc[ps_mask, 'n_temporal_patches'].fillna(0).sum())
total_chan = float(df_audit.loc[ps_mask, 'n_channel_patches' ].fillna(0).sum())

def _w(v, total):
    if v is None or pd.isna(v) or not total: return None
    return round(float(v) / float(total), 6)

df_audit['dataset_weight_temporal_patches'] = df_audit.apply(
    lambda r: _w(r['n_temporal_patches'], total_temp) if r['role'] == 'PRETRAIN_SOURCE' else None,
    axis=1,
)
df_audit['dataset_weight_channel_patches'] = df_audit.apply(
    lambda r: _w(r['n_channel_patches'], total_chan) if r['role'] == 'PRETRAIN_SOURCE' else None,
    axis=1,
)

df_audit = df_audit.sort_values(['role', 'dominio', 'dataset']).reset_index(drop=True)


# ---------------------------------------------------------------------------
# 3. Asserts duros ANTES de escribir el CSV. Si algo cambia silenciosamente,
#    abortamos aqui y no se sobreescribe ningun artefacto.
# ---------------------------------------------------------------------------
print()
print('--- Asserts de cierre v2.3 (antes de escribir CSV) ---')
expected_roles = {'PRETRAIN_SOURCE': 36, 'TRANSFER_TARGET': 11, 'DROP': 4, 'EXCLUDED': 2}
actual_roles   = df_audit['role'].value_counts().to_dict()
assert actual_roles == expected_roles, (
    f'Roles cambiaron: actual={actual_roles}, esperado={expected_roles}. '
    f'Revisar decidir_role o la particion manual. CSV no se sobreescribe.'
)
print(f'  roles {actual_roles}  OK')

cmapss = df_audit[df_audit['dataset'] == 'CMAPSS']
assert len(cmapss) == 1, 'CMAPSS no encontrado o duplicado'
cmapss = cmapss.iloc[0]
assert cmapss['role'] == 'TRANSFER_TARGET', f'CMAPSS role={cmapss["role"]}'
assert cmapss['evaluation_tier'] == 'primary', f'CMAPSS tier={cmapss["evaluation_tier"]}'
print(f'  CMAPSS = TRANSFER_TARGET primary  OK')

tt_actual   = set(df_audit.loc[df_audit['role'] == 'TRANSFER_TARGET', 'dataset'].tolist())
tt_expected = set(TRANSFER_TARGETS_PROPUESTOS.keys())
assert tt_actual == tt_expected, (
    f'TT difiere de TRANSFER_TARGETS_PROPUESTOS: extra={tt_actual - tt_expected}, '
    f'falta={tt_expected - tt_actual}'
)
print(f'  TT == TRANSFER_TARGETS_PROPUESTOS  OK ({len(tt_actual)} datasets)')

pad_extremo = df_audit[
    (df_audit['role'].isin(['PRETRAIN_SOURCE', 'TRANSFER_TARGET'])) &
    (pd.to_numeric(df_audit['padding_ratio'], errors='coerce') > UMBRAL_DROP_PADDING_RATIO)
]
assert pad_extremo.empty, (
    f'Padding > {UMBRAL_DROP_PADDING_RATIO} en PS/TT: {pad_extremo["dataset"].tolist()}. '
    f'Esto no deberia pasar con pad salvo bug.'
)
print(f'  ningun PS/TT con padding > {UMBRAL_DROP_PADDING_RATIO}  OK')

ok_rows = df_audit[df_audit['status'] == 'ok']
versions_ok = ok_rows['audit_version'].dropna().unique().tolist()
assert versions_ok == [AUDIT_VERSION], f'audit_version distinto de {AUDIT_VERSION}: {versions_ok}'
tail_ok = ok_rows['tail_policy'].dropna().unique().tolist()
assert tail_ok == [TAIL_POLICY], f'tail_policy distinto de {TAIL_POLICY}: {tail_ok}'
print(f'  audit_version={AUDIT_VERSION} y tail_policy={TAIL_POLICY} en todas las filas ok')
print('--- Asserts PASS ---')
print()


# ---------------------------------------------------------------------------
# 4. Escritura ATOMICA del CSV: tmp + replace. Solo despues de PASS.
# ---------------------------------------------------------------------------
tmp_csv = REPORT_CSV.with_suffix('.csv.tmp')
df_audit.to_csv(tmp_csv, index=False)
tmp_csv.replace(REPORT_CSV)
print(f'CSV guardado (atomico): {REPORT_CSV} ({len(df_audit)} filas, audit_version={AUDIT_VERSION})')


# ---------------------------------------------------------------------------
# 5. Diff v2.2 -> v2.3 con GUARD: solo si el snapshot previo es v2.2 y el
#    actual es v2.3. Cualquier otra combinacion (snapshot ya v2.3, sin
#    snapshot, etc.) imprime una nota y no escribe el diff.
# ---------------------------------------------------------------------------
print()
print('--- Diff v2.2 -> v2.3 ---')
curr_v = df_audit['audit_version'].iloc[0] if 'audit_version' in df_audit.columns else None
generar_diff = (df_audit_previous is not None
                and prev_version_str == '2.2'
                and str(curr_v) == '2.3')

if not generar_diff:
    if df_audit_previous is None:
        motivo = 'no hay snapshot previo'
    elif prev_version_str != '2.2':
        motivo = f'snapshot previo es v={prev_version_str}, no v2.2'
    elif str(curr_v) != '2.3':
        motivo = f'audit actual es v={curr_v}, no v2.3'
    else:
        motivo = 'desconocido'
    print(f'  diff omitido: {motivo}.')
    print(f'  (no se sobreescribe tail_policy_diff_v22_v23.*)')
else:
    prev = df_audit_previous.set_index('dataset')
    curr = df_audit.set_index('dataset')
    common = sorted(prev.index.intersection(curr.index))

    def _num(v):
        try:
            return float(v)
        except (TypeError, ValueError):
            return None

    diff_rows = []
    for ds in common:
        p = prev.loc[ds]
        c = curr.loc[ds]
        if c['role'] not in ('PRETRAIN_SOURCE', 'TRANSFER_TARGET'):
            continue
        nw_p,  nw_c  = _num(p.get('n_windows')),        _num(c.get('n_windows'))
        pad_p, pad_c = _num(p.get('padding_ratio')),    _num(c.get('padding_ratio'))
        cp_p,  cp_c  = _num(p.get('n_channel_patches')), _num(c.get('n_channel_patches'))
        sz_p,  sz_c  = _num(p.get('estimated_size_mb')), _num(c.get('estimated_size_mb'))
        delta_w      = (nw_c - nw_p) if (nw_p is not None and nw_c is not None) else None
        delta_cp_pct = (100.0 * (cp_c - cp_p) / cp_p) if (cp_p and cp_c) else None
        diff_rows.append({
            'dataset':                 ds,
            'role':                    c['role'],
            'n_windows_v22':           nw_p,
            'n_windows_v23':           nw_c,
            'delta_windows':           delta_w,
            'padding_v22':             pad_p,
            'padding_v23':             pad_c,
            'n_channel_patches_v22':   cp_p,
            'n_channel_patches_v23':   cp_c,
            'delta_channel_patches_pct': round(delta_cp_pct, 2) if delta_cp_pct is not None else None,
            'estimated_size_mb_v22':   sz_p,
            'estimated_size_mb_v23':   sz_c,
        })

    df_diff = pd.DataFrame(diff_rows).sort_values('delta_channel_patches_pct', ascending=False, na_position='last')
    diff_csv  = REPORT_DIR / 'tail_policy_diff_v22_v23.csv'
    diff_json = REPORT_DIR / 'tail_policy_diff_v22_v23.json'
    # Escritura atomica
    tmp_diff_csv = diff_csv.with_suffix('.csv.tmp')
    df_diff.to_csv(tmp_diff_csv, index=False)
    tmp_diff_csv.replace(diff_csv)
    tmp_diff_json = diff_json.with_suffix('.json.tmp')
    tmp_diff_json.write_text(json.dumps({
        'from_audit_version': prev_version_str,
        'to_audit_version':   str(curr_v),
        'timestamp':          datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'tail_policy_change': 'drop -> pad',
        'window_size': WINDOW_SIZE, 'stride': STRIDE, 'patch_size': PATCH_SIZE,
        'datasets':           df_diff.to_dict(orient='records'),
    }, indent=2, ensure_ascii=False, default=str))
    tmp_diff_json.replace(diff_json)
    print(f'  diff generado: {diff_csv}')
    print(f'                  {diff_json}')
    print()
    print('Top 10 datasets por delta_channel_patches_pct:')
    print(df_diff.head(10).to_string(index=False))
    print()
    print('Bottom 5 datasets por delta_channel_patches_pct:')
    print(df_diff.tail(5).to_string(index=False))

print()
print('Resumen rapido del CSV v2.3:')
df_audit[['dataset', 'role', 'audit_version', 'tail_policy', 'subset_id_col_detected',
          'n_canales', 'n_unidades_total', 'n_temporal_patches',
          'n_channel_patches', 'n_dense_channel_patches',
          'invalid_patch_ratio', 'dataset_weight_channel_patches']].head(15)


## 17. Reporte markdown

Reporte legible con secciones por role. Pensado para incluir como apendice en la memoria del TFM.

In [ ]:
from collections import Counter


def render_md(df):
    L = []
    L.append(f'# Auditoria de datasets PHMD (v{AUDIT_VERSION})')
    L.append('')
    L.append('Generado por `notebooks/exploration/02_dataset_audit.ipynb`. '
             'Las decisiones son heuristicas y deben revisarse manualmente para los casos limite.')
    L.append('')
    L.append('## Cambios v2.3 frente a v2.2')
    L.append('')
    L.append("- `tail_policy='pad'` adoptado como politica operacional global. La ventana extra "
             'parcial con padding por la derecha sustituye al descarte de cola del modo `drop`. '
             'La justificacion esta en `results/audit/tail_policy_comparison.json` (comparativa '
             'sobre los 7 datasets con `tail_drop_ratio > 0.05` en v2.2): PHM18 como TT primary '
             'perdia el 23.2% de sus filas crudas con `drop`. Las mascaras `valid_time_mask` y '
             '`valid_patch_mask` ya forman parte del contrato y absorben el padding anadido. '
             'El diff agregado v2.2 -> v2.3 esta en `results/audit/tail_policy_diff_v22_v23.csv`.')
    L.append('- Conteo exacto de patches validos por ventana (sin derivar de `padding_ratio`).')
    L.append('- Metricas densas nuevas: `n_dense_temporal_patches`, `n_dense_channel_patches`, '
             '`invalid_patch_ratio`, `dense_vs_valid_ratio`. Permiten distinguir senal valida '
             'del coste denso real del encoder channel-independent.')
    L.append('- `audit_summary.json` incluye `tail_policy` top-level y un bloque '
             '`tail_policy_decision` autocontenido con la justificacion.')
    L.append('- `sampling_policy` con valores iniciales numericos: '
             '`cap_max_dataset_weight=0.10`, `cap_max_client_weight=0.25`, '
             '`min_client_presence=0.005`.')
    L.append('- Warning nuevo `tail_policy_pad_padding_moderado` para TT con padding entre 0.15 '
             'y 0.5 (caso esperado: PHM18 y PHMAP23 tras pad).')
    L.append('- Asserts de cierre tras agregar: roles, CMAPSS=TT primary, lista TT exacta y '
             'ausencia de padding extremo en PS/TT. La generacion del CSV falla si algo cambia '
             'silenciosamente.')
    L.append('')
    L.append('### Heredado de v2.2')
    L.append('')
    L.append("- `decidir_role` respeta `TRANSFER_TARGETS_PROPUESTOS` antes de aplicar los DROP "
             'por longitud o numero de ventanas (las reglas duras `nan_pct_max` y '
             '`padding_ratio > 0.8` siguen aplicando a todos).')
    L.append('- `estimated_n_shards = suma de techos por split` (alineado con la escritura real).')
    L.append('- `audit_groups.json` con estructura `{audit_version, timestamp, window_size, '
             'stride, patch_size, tail_policy, clients}`.')
    L.append('- Overrides manuales: `SUBSET_ID_OVERRIDE={CMAPSS:FD}`, '
             '`TARGET_COL_OVERRIDE={UNIBO21:soc}`. Trayectorias agrupadas por '
             '`split + subset_id + unit_col`.')
    L.append('- `sampling_rate_info` agregado en `audit_summary.json` (no warning por dataset).')
    L.append('')
    L.append('## Nota sobre patches temporales, por canal y densos')
    L.append('')
    L.append('- `n_temporal_patches` = patches validos por ventana, agregado.')
    L.append('- `n_channel_patches`  = `n_temporal_patches * n_canales`. Base para coste/dominancia.')
    L.append('- `n_dense_temporal_patches` = `n_windows * N_PATCHES` (sin descontar padding).')
    L.append('- `n_dense_channel_patches`  = `n_dense_temporal_patches * n_canales`.')
    L.append('- `invalid_patch_ratio`      = `1 - n_channel_patches / n_dense_channel_patches`.')
    L.append('- `dense_vs_valid_ratio`     = `n_dense_channel_patches / n_channel_patches`.')
    L.append('')
    L.append('Con `tail_policy=pad`, el coste denso puede subir mas que los patches validos. '
             'La loss SSL debe enmascarar tanto el patch invalido (`valid_patch_mask`) como el '
             'padding dentro de un patch parcialmente valido (`valid_time_mask.reshape(N, P)`).')
    L.append('')
    rec = Counter(df['role'].fillna('?'))
    dom = Counter(df['dominio'].fillna('?'))
    L.append('## Resumen')
    L.append('')
    L.append(f'- Datasets en el reporte: **{len(df)}**')
    L.append('- Por role:')
    for k, v in sorted(rec.items()):
        L.append(f'  - `{k}`: {v}')
    L.append('- Por dominio:')
    for k, v in sorted(dom.items()):
        L.append(f'  - `{k}`: {v}')
    L.append('')
    L.append('## Tabla resumen')
    L.append('')
    cols = ['dataset', 'dominio', 'role', 'evaluation_tier',
            'subset_id_col_detected', 'tail_policy',
            'n_unidades_total', 'n_canales', 'len_mediana',
            'n_windows', 'padding_ratio',
            'n_channel_patches', 'n_dense_channel_patches', 'invalid_patch_ratio',
            'tipo_target']
    sub = df[cols].fillna('')
    L.append('| ' + ' | '.join(cols) + ' |')
    L.append('|' + '|'.join(['---'] * len(cols)) + '|')
    for _, row in sub.iterrows():
        L.append('| ' + ' | '.join(str(row[c]) for c in cols) + ' |')
    L.append('')

    # Diff v2.2 -> v2.3: solo se incluye en el reporte si la pasada actual lo
    # ha generado (cell 34 setea `generar_diff = True` cuando snapshot previo
    # es v2.2 y audit actual es v2.3). En cualquier otro caso, evitamos
    # arrastrar un diff antiguo como si fuese de esta ejecucion.
    diff_path = REPORT_DIR / 'tail_policy_diff_v22_v23.csv'
    diff_json_path = REPORT_DIR / 'tail_policy_diff_v22_v23.json'
    incluir_diff = False
    diff_motivo_skip = None
    try:
        if generar_diff:
            incluir_diff = True
        else:
            diff_motivo_skip = 'snapshot previo no era v2.2 o audit actual no es v2.3'
    except NameError:
        diff_motivo_skip = '`generar_diff` no definido en este scope'

    if incluir_diff and diff_path.exists():
        try:
            df_diff = pd.read_csv(diff_path)
            # Validacion extra: el JSON del diff debe corresponder a v2.2 -> v2.3.
            from_v, to_v = None, None
            if diff_json_path.exists():
                try:
                    meta_diff = json.loads(diff_json_path.read_text())
                    from_v = str(meta_diff.get('from_audit_version'))
                    to_v   = str(meta_diff.get('to_audit_version'))
                except Exception:
                    pass
            if (from_v, to_v) != ('2.2', '2.3'):
                L.append('## Diff v2.2 -> v2.3 omitido')
                L.append('')
                L.append(f'_El fichero `tail_policy_diff_v22_v23.json` no corresponde a esta '
                         f'transicion (from={from_v}, to={to_v}). No se incluye en el reporte._')
                L.append('')
            else:
                L.append('## Diff v2.2 (drop) -> v2.3 (pad) - top 10 deltas')
                L.append('')
                L.append('Esta es la lectura agregada de los cambios en metricas tras pasar a '
                         "`tail_policy='pad'`. La tabla detallada esta en "
                         '`results/audit/tail_policy_diff_v22_v23.csv` y la justificacion '
                         '(sobre los 7 datasets con `tail_drop_ratio_alto`) en '
                         '`results/audit/tail_policy_comparison.json`.')
                L.append('')
                top = df_diff.sort_values('delta_channel_patches_pct', ascending=False, na_position='last').head(10)
                cols_diff = ['dataset', 'role', 'n_windows_v22', 'n_windows_v23', 'delta_windows',
                              'padding_v22', 'padding_v23',
                              'n_channel_patches_v22', 'n_channel_patches_v23',
                              'delta_channel_patches_pct']
                L.append('| ' + ' | '.join(cols_diff) + ' |')
                L.append('|' + '|'.join(['---'] * len(cols_diff)) + '|')
                for _, r in top.iterrows():
                    L.append('| ' + ' | '.join(str(r.get(c, '')) for c in cols_diff) + ' |')
                L.append('')
        except Exception as _e:
            L.append(f'_(no se pudo leer diff: {_e})_')
            L.append('')
    else:
        L.append('## Diff v2.2 -> v2.3 omitido en esta ejecucion')
        L.append('')
        motivo = diff_motivo_skip or 'el fichero del diff no existe en disco'
        L.append(f'_{motivo}. No se incluye un diff antiguo como si fuese de esta pasada._')
        L.append('')

    orden = ['PRETRAIN_SOURCE', 'TRANSFER_TARGET', 'DROP', 'EXCLUDED']
    for role in orden:
        sub = df[df['role'] == role]
        if len(sub) == 0: continue
        L.append(f'## {role} ({len(sub)} datasets)')
        L.append('')
        for _, row in sub.iterrows():
            L.append(f'### {row["dataset"]}  ·  dominio: `{row["dominio"]}`')
            L.append(f'- Razon: {row["role_razon"]}')
            if pd.notna(row.get('evaluation_tier')) and row['evaluation_tier']:
                L.append(f'- Evaluation tier: `{row["evaluation_tier"]}`')
            if pd.notna(row.get('subset_id_col_detected')) and row['subset_id_col_detected']:
                L.append(f'- Subset id col detectado: `{row["subset_id_col_detected"]}` (excluido de canales)')
            if pd.notna(row['n_unidades_total']):
                L.append(f'- Forma: {row["n_unidades_total"]} trayectorias, '
                         f'{row["n_canales"]} canales, longitud mediana {row["len_mediana"]}')
            if pd.notna(row['n_windows']):
                L.append(f'- Ventaneo (W={WINDOW_SIZE}, tail={row.get("tail_policy") or "drop"}): '
                         f'{row["n_windows"]} ventanas, '
                         f'padding_ratio={row["padding_ratio"]}, '
                         f'~{row["n_temporal_patches"]} temporal patches, '
                         f'~{row["n_channel_patches"]} channel patches validos, '
                         f'~{row["n_dense_channel_patches"]} densos '
                         f'(invalid_patch_ratio={row.get("invalid_patch_ratio")}), '
                         f'~{row["estimated_size_mb"]} MB, '
                         f'~{row["estimated_n_shards"]} shards')
            if pd.notna(row['sampling_rate_hz']):
                L.append(f'- Temporal: sampling_rate={row["sampling_rate_hz"]} Hz '
                         f'({row["sampling_rate_origen"]}), '
                         f'window_time_seconds={row["window_time_seconds"]}')
            else:
                L.append('- Temporal: sampling_rate desconocido; W=512 son muestras, no duracion fisica.')
            if pd.notna(row['target_col']):
                L.append(f'- Target: `{row["target_col"]}` ({row["tipo_target"]}, '
                         f'policy={row["target_policy"]}); cobertura {row["label_coverage_pct"]}%')
                if row.get('target_candidates'):
                    L.append(f'- Target candidates: `{row["target_candidates"]}`')
            if pd.notna(row['n_clases']) and 'classification' in str(row['tipo_target']):
                L.append(f'- Clases: {row["n_clases"]}, balance {row["balance_clases"]}')
            elif row['tipo_target'] in ('rul', 'regression', 'regression_or_rul'):
                L.append(f'- Rango target: [{row["rul_min"]}, {row["rul_max"]}]')
            if pd.notna(row.get('dataset_weight_channel_patches')):
                L.append(f'- Peso en corpus PS (channel patches): '
                         f'{row["dataset_weight_channel_patches"]*100:.2f}%')
            if pd.notna(row['plot']):
                L.append(f'- Plot: `{row["plot"]}`')
            L.append('')
    return '\n'.join(L)

md = render_md(df_audit)
REPORT_MD.write_text(md)
print(f'Markdown guardado: {REPORT_MD} ({len(md)} caracteres, audit_version={AUDIT_VERSION})')


## 18. Agrupacion por cliente FL

Solo los PRETRAIN_SOURCE forman parte del corpus federado. La agrupacion natural es por dominio. Los dominios con un unico PRETRAIN_SOURCE pequeno se absorben en un cliente `misc_industrial` para evitar clientes minusculos. El resultado se serializa en `audit_groups.json` para alimentar el sampler federado en pasos posteriores.

In [ ]:
import json
from collections import defaultdict
from datetime import datetime, timezone
import pandas as pd

# Mantenemos la fusion existente. La regla `no fusionar mas clientes pequenos`
# se respeta: wind/cnc_milling/hdd siguen como clientes independientes.
DOMINIOS_FUSIONABLES = {
    'compressor', 'drills', 'transformers', 'capacitors',
    'building_hvac', 'mosfets_power', 'naval', 'learning_curves',
}
CLIENTE_FUSION = 'misc_industrial'


def _entero_seguro(x):
    if x is None or (isinstance(x, float) and pd.isna(x)): return 0
    try: return int(x)
    except (TypeError, ValueError): return 0


# Pasada 1: contar PS por dominio para decidir fusiones
ps_por_dominio = defaultdict(list)
for _, row in df_audit.iterrows():
    if row['role'] != 'PRETRAIN_SOURCE': continue
    ps_por_dominio[row['dominio']].append(row['dataset'])

fusionar = {d for d, ds in ps_por_dominio.items()
            if d in DOMINIOS_FUSIONABLES and len(ds) == 1}
print(f'Dominios fusionados a {CLIENTE_FUSION}: {sorted(fusionar)}')

# Totales del corpus para los pesos por cliente
ps_mask     = df_audit['role'] == 'PRETRAIN_SOURCE'
total_temp  = float(df_audit.loc[ps_mask, 'n_temporal_patches'].fillna(0).sum())
total_chan  = float(df_audit.loc[ps_mask, 'n_channel_patches' ].fillna(0).sum())

agrupados = defaultdict(list)
for _, row in df_audit.iterrows():
    if row['role'] != 'PRETRAIN_SOURCE': continue
    dom = row['dominio']
    cliente = CLIENTE_FUSION if dom in fusionar else dom
    agrupados[cliente].append({
        'dataset':            row['dataset'],
        'role':               row['role'],
        'dominio':            dom,
        'n_unidades':         _entero_seguro(row['n_unidades_total']),
        'n_windows':          _entero_seguro(row['n_windows']),
        'n_temporal_patches': _entero_seguro(row['n_temporal_patches']),
        'n_channel_patches':  _entero_seguro(row['n_channel_patches']),
        'size_mb':            float(row['estimated_size_mb']) if pd.notna(row['estimated_size_mb']) else 0.0,
    })

# Ordenamos cada cliente por channel_patches (descendente)
for k in agrupados:
    agrupados[k].sort(key=lambda x: x['n_channel_patches'], reverse=True)

# Pesos por cliente sobre el corpus total
client_weights = {}
for cliente, ds_list in agrupados.items():
    sum_temp = sum(x['n_temporal_patches'] for x in ds_list)
    sum_chan = sum(x['n_channel_patches']  for x in ds_list)
    client_weights[cliente] = {
        'n_datasets':                       len(ds_list),
        'n_temporal_patches':               sum_temp,
        'n_channel_patches':                sum_chan,
        'client_weight_temporal_patches':   round(sum_temp / total_temp, 6) if total_temp else 0.0,
        'client_weight_channel_patches':    round(sum_chan / total_chan, 6) if total_chan else 0.0,
    }

# Escribimos audit_groups.json con metadata + datasets agregados por cliente.
# v2.3: estructura {audit_version, timestamp, window_size, stride, patch_size,
# tail_policy, clients}. Los consumidores deben leer payload['clients'] para el detalle.
# El note documenta el contrato explicito para evitar codigo que asuma el
# formato antiguo {cliente: {...}}.
groups_clients = {
    cliente: {
        **client_weights[cliente],
        'datasets': agrupados[cliente],
    }
    for cliente in agrupados
}
groups_payload = {
    'audit_version': AUDIT_VERSION,
    'timestamp':     datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'window_size':   WINDOW_SIZE,
    'stride':        STRIDE,
    'patch_size':    PATCH_SIZE,
    'tail_policy':   TAIL_POLICY,
    'note':          'Estructura v2.3: leer payload["clients"][cliente] para el detalle '
                     'por cliente. La meta top-level expone audit_version, timestamp, '
                     'window_size, stride, patch_size y tail_policy operacional.',
    'clients':       groups_clients,
}
GROUPS_JSON.write_text(json.dumps(groups_payload, indent=2, ensure_ascii=False))
print(f'audit_groups.json guardado: {GROUPS_JSON} (audit_version={AUDIT_VERSION}, tail_policy={TAIL_POLICY})')
print()
print('Resumen por cliente FL (solo PRETRAIN_SOURCE):')
print(f'  {"cliente":18s} {"n_ds":>4s} {"unidades":>10s} {"ventanas":>10s} {"temp_patch":>14s} {"chan_patch":>14s} {"w_temp%":>9s} {"w_chan%":>9s}')
for cliente in sorted(agrupados, key=lambda c: -client_weights[c]['n_channel_patches']):
    info = client_weights[cliente]
    ds = agrupados[cliente]
    print(f'  {cliente:18s} {info["n_datasets"]:>4d} '
          f'{sum(x["n_unidades"] for x in ds):>10d} '
          f'{sum(x["n_windows"] for x in ds):>10d} '
          f'{info["n_temporal_patches"]:>14,d} '
          f'{info["n_channel_patches"]:>14,d} '
          f'{info["client_weight_temporal_patches"]*100:>9.2f} '
          f'{info["client_weight_channel_patches"]*100:>9.2f}')

## 19. Sanity checks y audit_summary

Verificamos consistencia de conteos: `descargados_ok = EXCLUDED + auditados`. Calculamos pesos por cliente FL (% sobre tokens totales) y emitimos warning si alguno supera `UMBRAL_DOMINANCIA_CLIENTE` (regla anti-dominancia, `CLAUDE.md` sec. 7).

In [ ]:
import json
from collections import Counter

n_descargados = len(descargados_ok)
n_excluded    = len(EXCLUIDOS_PERMANENTES)
auditados     = df_audit[df_audit['status'].isin(['ok', 'failed'])]
n_auditados   = len(auditados)
rec_counts    = Counter(df_audit['role'].fillna('?'))
n_filas_csv   = len(df_audit)

# Top datasets por tres metricas
ps_df = df_audit[df_audit['role'] == 'PRETRAIN_SOURCE'].copy()
top_temp = ps_df.nlargest(10, 'n_temporal_patches')[['dataset', 'n_temporal_patches',
                                                       'dataset_weight_temporal_patches']].to_dict('records')
top_chan = ps_df.nlargest(10, 'n_channel_patches' )[['dataset', 'n_channel_patches',
                                                       'dataset_weight_channel_patches']].to_dict('records')
top_size = ps_df.nlargest(10, 'estimated_size_mb' )[['dataset', 'estimated_size_mb']].to_dict('records')
top_dense = ps_df.nlargest(10, 'n_dense_channel_patches')[['dataset', 'n_dense_channel_patches',
                                                            'invalid_patch_ratio']].to_dict('records')

# Sampling policy v2.3: valores numericos iniciales
sampling_policy = {
    'level':                  'dataset_and_client_balanced',
    'base_weight':            'n_channel_patches',
    'cap_max_dataset_weight': 0.10,
    'cap_max_client_weight':  0.25,
    'min_client_presence':    0.005,
    'notes': (
        'Valores iniciales para el sampler de pretraining; ajustables tras inspeccion '
        'de batches reales. Limita PHM10 (~22.7% del corpus en channel patches) y '
        'evita que cnc_milling (~0.10%) desaparezca del entrenamiento federado. '
        'Centralizado y federado deben usar politicas equivalentes para que la '
        'comparacion sea justa.'
    ),
}

# Decision tail_policy autocontenida
tail_policy_decision = {
    'decision':                 'pad_global',
    'source':                   'tail_policy_comparison.json',
    'main_reason':              'PHM18 TT primary lost 23.2% raw rows under drop',
    'decided_in_audit_version': AUDIT_VERSION,
    'diff_artifact':            'tail_policy_diff_v22_v23.csv',
}

# Informacion agregada sobre sampling_rate
relevantes  = df_audit[df_audit['role'].isin(['PRETRAIN_SOURCE', 'TRANSFER_TARGET'])]
sr_unknown  = relevantes[relevantes['sampling_rate_hz'].isna()]
sr_inferred = relevantes[
    relevantes['sampling_rate_origen'].fillna('').astype(str) == 'inferido'
]
sampling_rate_info = {
    'relevant_count':                int(len(relevantes)),
    'known_count':                   int(len(relevantes) - len(sr_unknown)),
    'unknown_count':                 int(len(sr_unknown)),
    'unknown_datasets':              sorted(sr_unknown['dataset'].tolist()),
    'inferred_provisional_datasets': sorted(sr_inferred['dataset'].tolist()),
    'note': ('W=512 son muestras, no duracion fisica. Los datasets en '
             'inferred_provisional_datasets tienen sampling_rate calculado a partir de '
             'deltas temporales pero no se han validado los checks de monotonicidad '
             'pre/post ordenamiento exigidos en CLAUDE.md sec.5. Tratar como provisional.'),
}

# Totales densos PS
total_dense_temp = int(df_audit.loc[ps_df.index, 'n_dense_temporal_patches'].fillna(0).sum())
total_dense_chan = int(df_audit.loc[ps_df.index, 'n_dense_channel_patches'].fillna(0).sum())

# Warnings automaticos
advertencias = []

if n_excluded + n_auditados != n_descargados:
    advertencias.append({
        'tipo': 'inconsistencia_conteos',
        'detalle': f'{n_descargados} descargados != {n_excluded} EXCLUDED + {n_auditados} auditados',
    })
if n_filas_csv != n_descargados:
    advertencias.append({
        'tipo': 'inconsistencia_csv',
        'detalle': f'{n_filas_csv} filas en CSV != {n_descargados} descargados',
    })

TIPOS_RUL = {'rul', 'regression_or_rul'}
for _, row in df_audit.iterrows():
    nombre = row['dataset']
    role   = row['role']
    tipo_target = row.get('tipo_target') or ''

    if role == 'TRANSFER_TARGET':
        if pd.notna(row['n_windows']) and row['n_windows'] < 100:
            advertencias.append({
                'tipo': 'tt_n_windows_bajo', 'dataset': nombre,
                'detalle': f'{int(row["n_windows"])} ventanas (< 100)',
            })
        if pd.notna(row['n_unidades_total']) and row['n_unidades_total'] < 5:
            advertencias.append({
                'tipo': 'tt_n_units_bajo', 'dataset': nombre,
                'detalle': f'{int(row["n_unidades_total"])} unidades (< 5)',
            })

    if pd.notna(row['balance_clases']) and float(row['balance_clases']) < 0.05:
        advertencias.append({
            'tipo': 'class_balance_bajo', 'dataset': nombre,
            'detalle': f'balance_clases={row["balance_clases"]} (< 0.05)',
        })

    if pd.notna(row['rul_min']) and float(row['rul_min']) < 0:
        if tipo_target in TIPOS_RUL:
            advertencias.append({
                'tipo': 'rul_negative_values', 'dataset': nombre,
                'detalle': f'rul_min={row["rul_min"]} (target tipo_target={tipo_target})',
            })
        elif tipo_target == 'regression':
            advertencias.append({
                'tipo': 'negative_target_values', 'dataset': nombre,
                'detalle': f'target_min={row["rul_min"]} (target tipo_target={tipo_target}, no es RUL)',
            })

    if pd.notna(row['padding_ratio']) and float(row['padding_ratio']) > 0.15:
        advertencias.append({
            'tipo': 'padding_ratio_alto', 'dataset': nombre,
            'detalle': f'padding_ratio={row["padding_ratio"]} (> 0.15)',
        })

    if role == 'TRANSFER_TARGET' and pd.notna(row['padding_ratio']) and float(row['padding_ratio']) > 0.5:
        advertencias.append({
            'tipo': 'tt_padding_alto', 'dataset': nombre,
            'detalle': f'padding_ratio={row["padding_ratio"]} > 0.5 con W={WINDOW_SIZE}',
        })

    # Warning nuevo v2.3: padding moderado en TT tras pasar a pad.
    # Padding entre 0.15 y 0.5 indica que la cola de la trayectoria es
    # comparable al tamano de ventana. Aceptado porque las mascaras lo
    # absorben, pero conviene saberlo (PHM18 y PHMAP23 son los candidatos
    # esperables tras `pad`).
    if (role == 'TRANSFER_TARGET'
            and pd.notna(row['padding_ratio'])
            and 0.15 < float(row['padding_ratio']) <= 0.5
            and str(row.get('tail_policy')) == 'pad'):
        advertencias.append({
            'tipo': 'tail_policy_pad_padding_moderado', 'dataset': nombre,
            'detalle': (f'padding_ratio={row["padding_ratio"]} en (0.15, 0.5] con '
                        f"tail_policy='pad' (TT). Aceptado porque valid_time_mask "
                        'y valid_patch_mask forman parte del contrato.'),
        })

    median_lt_w = str(row.get('median_length_lt_window_size') or '').lower() in ('true', '1', 'yes')
    if role == 'TRANSFER_TARGET' and median_lt_w:
        advertencias.append({
            'tipo': 'tt_median_lt_window', 'dataset': nombre,
            'detalle': f'len_mediana={row["len_mediana"]} < W={WINDOW_SIZE}',
        })

    if role in ('TRANSFER_TARGET', 'PRETRAIN_SOURCE'):
        n_filas = row.get('n_filas_total')
        tail    = row.get('dropped_tail_rows')
        if (n_filas is not None and tail is not None
                and pd.notna(n_filas) and pd.notna(tail) and float(n_filas) > 0):
            ratio_tail = float(tail) / float(n_filas)
            if ratio_tail > 0.05:
                advertencias.append({
                    'tipo': 'tail_drop_ratio_alto', 'dataset': nombre,
                    'detalle': (f'dropped_tail_rows/n_filas={ratio_tail:.3f} > 0.05 '
                                f'(tail_policy={row.get("tail_policy") or "drop"}). '
                                'Con tail_policy=pad este warning no deberia aparecer.'),
                })

    if pd.notna(row.get('dataset_weight_channel_patches')) and \
       float(row['dataset_weight_channel_patches']) > 0.20:
        advertencias.append({
            'tipo': 'dataset_dominante', 'dataset': nombre,
            'detalle': f'{row["dataset_weight_channel_patches"]*100:.1f}% del corpus (channel patches)',
        })

# Warnings por cliente (< 1% en ambas metricas)
for cliente, info in client_weights.items():
    if info['client_weight_temporal_patches'] < 0.01:
        advertencias.append({
            'tipo': 'cliente_pequeno_temporal', 'cliente': cliente,
            'detalle': f'{info["client_weight_temporal_patches"]*100:.3f}% del corpus (temporal patches)',
        })
    if info['client_weight_channel_patches'] < 0.01:
        advertencias.append({
            'tipo': 'cliente_pequeno_channel', 'cliente': cliente,
            'detalle': f'{info["client_weight_channel_patches"]*100:.3f}% del corpus (channel patches)',
        })

summary = {
    'audit_version':         AUDIT_VERSION,
    'tail_policy':           TAIL_POLICY,
    'tail_policy_decision':  tail_policy_decision,
    'timestamp':             datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'window_size':           WINDOW_SIZE,
    'stride':                STRIDE,
    'patch_size':            PATCH_SIZE,
    'shard_maxcount':        SHARD_MAXCOUNT,
    'descargados_ok':        n_descargados,
    'excluidos_phmd':        n_excluded,
    'auditados':             n_auditados,
    'filas_csv':             n_filas_csv,
    'roles':                 dict(rec_counts),
    'n_clientes_fl':         len(agrupados),
    'pesos_cliente':         client_weights,
    'total_temporal_patches_ps':       int(total_temp),
    'total_channel_patches_ps':        int(total_chan),
    'total_dense_temporal_patches_ps': total_dense_temp,
    'total_dense_channel_patches_ps':  total_dense_chan,
    'top_datasets_por_temporal_patches':       top_temp,
    'top_datasets_por_channel_patches':        top_chan,
    'top_datasets_por_dense_channel_patches':  top_dense,
    'top_datasets_por_size_mb':                top_size,
    'umbral_dominancia_cliente':       UMBRAL_DOMINANCIA_CLIENTE,
    'sampling_policy':                 sampling_policy,
    'sampling_rate_info':              sampling_rate_info,
    'advertencias':                    advertencias,
    'pendientes_revision_temporal': [
        'monotonicidad_en_orden_original',
        'monotonicidad_tras_ordenar',
        'sampling_rate_inferido_tras_ordenar',
    ],
}

SUMMARY_JSON.write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str))
print(f'audit_summary.json guardado: {SUMMARY_JSON} (audit_version={AUDIT_VERSION}, tail_policy={TAIL_POLICY})')
print()
for k in ('audit_version', 'tail_policy', 'window_size', 'stride', 'patch_size', 'shard_maxcount',
          'descargados_ok', 'excluidos_phmd', 'auditados', 'filas_csv',
          'roles', 'n_clientes_fl',
          'total_temporal_patches_ps', 'total_channel_patches_ps',
          'total_dense_temporal_patches_ps', 'total_dense_channel_patches_ps'):
    print(f'  {k:32s}: {summary[k]}')
print()
print('Tail policy decision:')
for k, v in tail_policy_decision.items():
    print(f'  {k:30s} {v}')
print()
print('Sampling policy (caps iniciales para pretraining):')
for k, v in sampling_policy.items():
    print(f'  {k:30s} {v}')
print()
print('Sampling rate (agregado):')
print(f'  relevantes (PS+TT)     : {sampling_rate_info["relevant_count"]}')
print(f'  conocidos              : {sampling_rate_info["known_count"]}')
print(f'  desconocidos           : {sampling_rate_info["unknown_count"]}')
print(f'  inferidos (provisional): {sampling_rate_info["inferred_provisional_datasets"]}')
print()
print(f'Warnings: {len(advertencias)}')
tipos = Counter(a['tipo'] for a in advertencias)
for t, n in sorted(tipos.items(), key=lambda x: -x[1]):
    print(f'  {t:36s} {n}')


## 20. Cierre

Si todo cuadra, los siguientes pasos del plan son:

1. Revisar el diff agregado `tail_policy_diff_v22_v23.csv` para confirmar que el cambio drop->pad afecta a los datasets esperados (PHM18 como caso decisivo, y el resto sin sorpresas).
2. Revisar manualmente los datasets en `DROP` para confirmar que la heuristica acierta. Los 4 actuales (AHU21, GENDEM18, PBHP16, MOSFET11) tienen razones estructurales documentadas.
3. Mantener CMAPSS como `TRANSFER_TARGET` primary con cautelas: `padding_ratio` alto, `len_mediana < W` y `rul_min < 0`. Los warnings automaticos ya lo registran.
4. Cerrar `sampling_policy` con los caps iniciales (0.10 / 0.25 / 0.005) en `audit_summary.json`; ajustar tras inspeccionar batches reales en pretraining.
5. Pasar a la harmonizacion (`04_harmonization_pilot.ipynb` para CMAPSS, `05_harmonization_full.ipynb` para los 47 datasets PS+TT) con `tail_policy='pad'`.

No lanzar `05_harmonization_full` hasta que el piloto CMAPSS v0.4 confirme la alineacion exacta con el audit v2.3.


In [ ]:
from collections import Counter

recs = Counter(df_audit['role'].fillna('?'))
print('Roles finales:')
for k, v in sorted(recs.items(), key=lambda x: -x[1]):
    print(f'  {k:18s} {v}')

print()
fallos = df_audit[df_audit['status'] == 'failed']
if len(fallos):
    print(f'Datasets que han fallado el audit ({len(fallos)}):')
    for _, row in fallos.iterrows():
        print(f'  - {row["dataset"]}: {row["error"]}')
else:
    print('Sin fallos en el bucle.')

print()
print('Outputs:')
print(f'  - {REPORT_CSV}')
print(f'  - {REPORT_MD}')
print(f'  - {GROUPS_JSON}')
print(f'  - {SUMMARY_JSON}')
print(f'  - {AUDIT_DIR} (JSONs y PNGs en Drive)')

print()
print('Changelog v2.3 (frente a v2.2):')
print('  + tail_policy=pad global como politica operacional.')
print('  + Metricas densas: n_dense_temporal_patches, n_dense_channel_patches,')
print('    invalid_patch_ratio, dense_vs_valid_ratio (en JSON, CSV, MD y summary PS).')
print('  + audit_summary.json con tail_policy top-level + tail_policy_decision block.')
print('  + sampling_policy con caps numericos (0.10 / 0.25 / 0.005).')
print('  + Snapshot v2.2 al inicio + asserts duros antes de escribir CSV +')
print('    diff agregado tail_policy_diff_v22_v23.csv/json al finalizar.')
print('  + Warning nuevo: tail_policy_pad_padding_moderado (TT con padding 0.15-0.5).')
print('  + Heredado de v2.2: dataset_weight_*, evaluation_tier, target_candidates,')
print('    sampling_rate_info agregado (no warning por dataset), estimated_n_shards')
print('    por split.')
print()
print('Pendiente (requeriria pasada parcial, no bloqueante):')
print('  - monotonicidad_en_orden_original / _tras_ordenar (afecta a datasets con time_col).')
print('  - sampling_rate_inferido_tras_ordenar.')
print('  - Marcado en audit_summary.pendientes_revision_temporal.')